# 🎵 Music Audio Inpainting Pipeline v2

Notebook ini mengimplementasikan pipeline hybrid SSL + Diffusion untuk music audio inpainting.

## Struktur Notebook
| Cell | Isi |
|------|-----|
| 1 | Cek GPU & Install dependencies |
| 2 | Mount Google Drive & setup folder |
| 3 | Download & Preprocessing MusicNet |
| 4 | Definisi FiLM Layer (improved init) |
| 5 | Definisi Fungsi Evaluasi (LSD gap-restricted dB, FAD, PEAQ_ODG) |
| 6 | Helper functions (memory management, checkpoint, group-aware split) |
| 6.5 | Dataset, DataLoader & Shared Utilities (mask, crossfade) |
| 6.6 | Training Loop (Reconstruction-based + CFG dropout) |
| 6.7 | Shared hybrid training helpers, shared decoder builders, checkpoint utilities |
| 7 | **BASELINE: CQT-Diff+ standalone TRAINED (tanpa encoder)** |
| 8A | Training CLAP + CQT-Diff+ |
| 8 | Evaluasi CLAP + CQT-Diff+ |
| 9A | Training CLAP + MAID |
| 9 | Evaluasi CLAP + MAID |
| 10A | Training AudioMAE + CQT-Diff+ |
| 10 | Evaluasi AudioMAE + CQT-Diff+ |
| 11A | Training AudioMAE + MAID |
| 11 | Evaluasi AudioMAE + MAID |
| 12 | Gabungkan & visualisasikan semua hasil |

## Model yang Dievaluasi
- **Baseline**: CQT-Diff+ (tanpa encoder SSL, tanpa FiLM)
- **Config 1**: CLAP + CQT-Diff+
- **Config 2**: CLAP + MAID
- **Config 3**: AudioMAE + CQT-Diff+
- **Config 4**: AudioMAE + MAID

## Alur Hybrid Terbaru
- Jalankan cell training hybrid terlebih dahulu untuk menghasilkan checkpoint best per kombinasi.
- Jika checkpoint sudah ada, cell training akan skip secara default kecuali `FORCE_RETRAIN = True`.
- Cell evaluasi hybrid selalu mencoba memuat checkpoint terlatih.
- Jika checkpoint hybrid belum ada, evaluasi akan berhenti dengan pesan yang jelas.

## Gap Duration yang Dievaluasi
100ms, 300ms, 500ms, 750ms, 1200ms, 1700ms

---
⚠️ **Pastikan Runtime → Change Runtime Type → GPU (T4) sebelum menjalankan!**

---
## CELL 1 — Cek GPU & Install Dependencies
**Jalankan cell ini pertama kali setiap sesi Colab baru.**

In [ ]:
# ============================================================
# CELL 1: CEK GPU & INSTALL DEPENDENCIES
# ============================================================
# Cara pakai: Jalankan sekali di awal setiap sesi.
# Estimasi waktu install: ~3-5 menit.
# ============================================================

import subprocess
import sys

# --- Cek GPU ---
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ GPU aktif: {gpu_name}")
        print(f"✅ VRAM tersedia: {vram:.1f} GB")
    else:
        print("❌ GPU tidak aktif! Pergi ke Runtime → Change Runtime Type → GPU")
        sys.exit()
except ImportError:
    print("PyTorch belum terinstall, melanjutkan instalasi...")

print("\n📦 Menginstall dependencies...")

packages = [
    "librosa",       # Load audio, CQT, Mel-spectrogram
    "soundfile",     # Baca/tulis file audio
    "audioread",     # Backend untuk librosa
    "transformers",  # Load CLAP dan AudioMAE dari HuggingFace
    "accelerate",    # Loading model besar lebih efisien
    "einops",        # Operasi tensor yang lebih mudah dibaca
    "timm",          # Library model vision, dipakai AudioMAE
    "tqdm",          # Progress bar
    "pandas",        # Simpan hasil evaluasi ke tabel
    "matplotlib",    # Visualisasi hasil
    "scipy",         # Operasi sinyal
    "numpy",         # Operasi array numerik
    "resampy",       # Resampling audio berkualitas tinggi
    "torchvggish",   # VGGish embeddings untuk FAD legacy
    "git+https://github.com/ashvala/AQUA-tk.git",  # AquaTK PEAQb untuk evaluasi PEAQ
    "visqol-lib-py", # Fallback perceptual quality metric jika PEAQb tidak tersedia
]

for pkg in packages:
    print(f"  Installing {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], capture_output=True)

# Clone repo CQT-Diff+ dari GitHub ke folder portabel.
# Repo resmi: https://github.com/eloimoliner/CQT_diff
# Vast.ai/local: gunakan PROJECT_ROOT (default: current working directory), bukan path Colab.
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", os.getcwd())
EXTERNAL_DIR = os.path.join(PROJECT_ROOT, "external")
CQT_DIFF_DIR = os.environ.get("CQT_DIFF_DIR", os.path.join(EXTERNAL_DIR, "CQT_diff"))
os.makedirs(EXTERNAL_DIR, exist_ok=True)

if os.path.exists(os.path.join(CQT_DIFF_DIR, ".git")):
    print(f"  CQT-Diff+ repository sudah ada: {CQT_DIFF_DIR}")
else:
    print(f"  Cloning CQT-Diff+ repository ke {CQT_DIFF_DIR}...")
    subprocess.run(
        ["git", "clone", "https://github.com/eloimoliner/CQT_diff.git", CQT_DIFF_DIR],
        check=False
    )

# Tambahkan repo ke Python path agar bisa di-import
if CQT_DIFF_DIR not in sys.path:
    sys.path.insert(0, CQT_DIFF_DIR)

print("\n✅ Semua dependencies berhasil diinstall!")
print("\n💡 Catatan: MAID belum punya repo publik resmi.")
print("   Pada Cell 9 & 11, implementasi MAID menggunakan versi replika")
print("   berdasarkan deskripsi di paper (gap-aware conditional diffusion).")

---
## CELL 2 — Mount Google Drive & Setup Folder

In [ ]:
# ============================================================
# CELL 2: MOUNT GOOGLE DRIVE & SETUP FOLDER
# ============================================================
# Cara pakai:
# - IS_LOCAL = True  → Jalankan di lokal (tidak perlu Google Drive)
# - IS_LOCAL = False → Jalankan di Google Colab dengan Google Drive
# ============================================================

import os

# ─── PARAMETER ───────────────────────────────────────────────
IS_LOCAL = True   # Vast.ai/local default. Ganti ke False hanya jika menggunakan Google Colab.
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", os.getcwd())
LOCAL_ROOT = os.environ.get("MUSIC_INPAINTING_ROOT", os.path.join(PROJECT_ROOT, "music_inpainting"))
# ─────────────────────────────────────────────────────────────

if IS_LOCAL:
    DATA_ROOT = LOCAL_ROOT
    print(f"💻 Mode lokal aktif. Root folder: {DATA_ROOT}")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/music_inpainting"
    print(f"☁️  Google Drive terpasang. Root folder: {DATA_ROOT}")

PATHS = {
    "dataset":      os.path.join(DATA_ROOT, "dataset"),
    "preprocessed": os.path.join(DATA_ROOT, "preprocessed"),
    "masked":       os.path.join(DATA_ROOT, "masked"),
    "outputs":      os.path.join(DATA_ROOT, "outputs"),
    "results":      os.path.join(DATA_ROOT, "results"),
    "checkpoints":  os.path.join(DATA_ROOT, "checkpoints"),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"✅ Folder '{name}': {path}")

# Buat subfolder output untuk setiap model (termasuk baseline)
ALL_MODELS = ["baseline_cqtdiff", "clap_cqtdiff", "clap_maid", "audiomae_cqtdiff", "audiomae_maid"]
for model in ALL_MODELS:
    os.makedirs(os.path.join(PATHS["outputs"], model), exist_ok=True)

print(f"\n✅ Semua folder siap!")
print(f"📁 Root folder: {DATA_ROOT}")


---
## CELL 3 — Download & Preprocessing MusicNet

⚠️ **Jalankan SEKALI saja.** Hasil disimpan ke Drive dan tidak perlu diulang.

In [ ]:
# ============================================================
# CELL 3: DOWNLOAD & PREPROCESSING MUSICNET
# ============================================================
# Cara pakai:
# - Jalankan SEKALI saja. Hasil disimpan ke Google Drive.
# - Set SKIP_IF_EXISTS = True jika preprocessing sudah pernah
#   dilakukan sebelumnya untuk menghemat waktu.
# - Estimasi waktu: ~15-30 menit
# ============================================================

import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm
import urllib.request
import random

# ============================================================
# KONFIGURASI
# ============================================================

# Set True jika preprocessing sudah pernah dijalankan
SKIP_IF_EXISTS = False

# Seed tetap untuk semua sampling dataset agar eksperimen reproducible
DATASET_RANDOM_SEED = 42

# Sample rate target: MusicNet native/CD quality, dibutuhkan PEAQ (44.1/48kHz)
TARGET_SR = 44100

# Panjang segmen audio (4 detik = cukup untuk gap 1700ms + konteks)
SEGMENT_DURATION = 4.0
SEGMENT_SAMPLES = int(TARGET_SR * SEGMENT_DURATION)  # 176400 samples

# Gap duration yang dievaluasi (dalam milidetik) — DIPERBARUI
GAP_DURATIONS_MS = [100, 300, 500, 750, 1200, 1700]

# Persentase dataset yang digunakan
DATASET_FRACTION = 0.03

# Jumlah segmen maksimal per lagu
MAX_SEGMENTS_PER_FILE = 5

# Metadata MusicNet dipakai untuk stratified sampling composer + instrument
MUSICNET_METADATA_URL = "https://zenodo.org/record/5120004/files/musicnet_metadata.csv"


# ============================================================
# FUNGSI DOWNLOAD
# ============================================================

def download_musicnet_metadata():
    """
    Download metadata MusicNet untuk mengambil label composer dan instrument/ensemble.
    Jika download gagal, pipeline tetap jalan dengan stratifikasi fallback "unknown".
    """
    dataset_dir = PATHS["dataset"]
    metadata_path = os.path.join(dataset_dir, "musicnet_metadata.csv")

    if os.path.exists(metadata_path):
        return pd.read_csv(metadata_path)

    try:
        print("📥 Downloading MusicNet metadata...")
        urllib.request.urlretrieve(MUSICNET_METADATA_URL, metadata_path)
        return pd.read_csv(metadata_path)
    except Exception as e:
        print(f"⚠️ Metadata MusicNet tidak bisa didownload ({e}).")
        print("   Stratified sampling akan fallback ke label composer/instrument='unknown'.")
        return pd.DataFrame()


def download_musicnet():
    """
    Download dataset MusicNet dari Zenodo.
    MusicNet berisi 330 rekaman musik klasik.
    Kita hanya pakai subset sesuai DATASET_FRACTION untuk efisiensi komputasi.
    """
    dataset_dir = PATHS["dataset"]
    audio_dir = os.path.join(dataset_dir, "audio")

    if os.path.exists(audio_dir) and len(os.listdir(audio_dir)) > 10:
        print("✅ Dataset sudah ada di Drive, skip download.")
        return audio_dir

    os.makedirs(audio_dir, exist_ok=True)
    MUSICNET_URL = "https://zenodo.org/record/5120004/files/musicnet.tar.gz"
    tar_path = os.path.join(dataset_dir, "musicnet.tar.gz")

    if os.path.exists(tar_path):
        print(f"Menggunakan archive MusicNet yang sudah ada: {tar_path}")
    else:
        print("?? Downloading MusicNet audio files...")
        print("   Estimasi ukuran: ~11GB full archive")

        def progress_hook(count, block_size, total_size):
            percent = min(count * block_size * 100 / total_size, 100)
            print(f"\r  Progress: {percent:.1f}%", end="")

        urllib.request.urlretrieve(MUSICNET_URL, tar_path, progress_hook)
        print("\n  Download selesai.")

    print("  Mengekstrak MusicNet audio...")
    import tarfile
    with tarfile.open(tar_path, "r:gz") as tar:
        wav_members = [m for m in tar.getmembers() if m.name.endswith('.wav')]
        for member in tqdm(wav_members, desc="Extracting"):
            tar.extract(member, audio_dir)

    print(f"? Dataset berhasil diekstrak ke {audio_dir}")
    return audio_dir


# ============================================================
# FUNGSI STRATIFIED SAMPLING
# ============================================================

def _track_id_from_path(audio_path):
    """Ambil MusicNet track id dari nama file audio, misal 1727.wav -> 1727."""
    return os.path.splitext(os.path.basename(audio_path))[0]


def _normalise_musicnet_metadata(metadata_df):
    """Rapikan nama kolom metadata agar robust terhadap variasi source."""
    if metadata_df is None or metadata_df.empty:
        return pd.DataFrame()

    meta = metadata_df.copy()
    meta.columns = [str(c).strip().lower() for c in meta.columns]

    if "id" not in meta.columns:
        return pd.DataFrame()

    meta["track_id"] = meta["id"].astype(str)
    if "composer" not in meta.columns:
        meta["composer"] = "unknown"

    # MusicNet metadata umum memakai ensemble; beberapa mirror menyediakan instrument.
    instrument_col = None
    for col in ["instrument", "instruments", "ensemble"]:
        if col in meta.columns:
            instrument_col = col
            break
    meta["instrument"] = meta[instrument_col] if instrument_col else "unknown"

    meta["composer"] = meta["composer"].fillna("unknown").astype(str)
    meta["instrument"] = meta["instrument"].fillna("unknown").astype(str)
    return meta[["track_id", "composer", "instrument"]].drop_duplicates("track_id")


def stratified_sample_dataframe(df, n_samples, stratify_cols, seed=DATASET_RANDOM_SEED):
    """
    Ambil sample deterministic dengan proporsi strata sebisa mungkin terjaga.
    Dipakai untuk stratifikasi composer + instrument pada pemilihan file dan eval.
    """
    if len(df) == 0 or n_samples <= 0:
        return df.iloc[0:0].copy()

    n_samples = min(int(n_samples), len(df))
    rng = np.random.default_rng(seed)

    work = df.copy().reset_index(drop=True)
    work["_sample_row_id"] = np.arange(len(work))
    for col in stratify_cols:
        if col not in work.columns:
            work[col] = "unknown"
        work[col] = work[col].fillna("unknown").astype(str)

    grouped = list(work.groupby(stratify_cols, dropna=False, sort=True))
    quotas = []
    for group_key, group in grouped:
        expected = len(group) * n_samples / len(work)
        base = int(np.floor(expected))
        quotas.append({
            "key": group_key,
            "group": group,
            "quota": min(base, len(group)),
            "fractional": expected - base,
            "tie": rng.random(),
        })

    remaining = n_samples - sum(q["quota"] for q in quotas)
    for q in sorted(quotas, key=lambda item: (-item["fractional"], item["tie"])):
        if remaining <= 0:
            break
        capacity = len(q["group"]) - q["quota"]
        if capacity > 0:
            q["quota"] += 1
            remaining -= 1

    sampled_parts = []
    for offset, q in enumerate(quotas):
        if q["quota"] > 0:
            sampled_parts.append(q["group"].sample(q["quota"], random_state=seed + offset))

    if sampled_parts:
        sampled = pd.concat(sampled_parts, ignore_index=True)
    else:
        sampled = work.iloc[0:0].copy()

    if len(sampled) < n_samples:
        missing = n_samples - len(sampled)
        sampled_ids = set(sampled["_sample_row_id"]) if "_sample_row_id" in sampled.columns else set()
        unsampled = work[~work["_sample_row_id"].isin(sampled_ids)]
        if len(unsampled) > 0:
            sampled = pd.concat([
                sampled,
                unsampled.sample(min(missing, len(unsampled)), random_state=seed + 999)
            ], ignore_index=True)

    return sampled.sample(frac=1.0, random_state=seed).drop(columns=["_sample_row_id"], errors="ignore").reset_index(drop=True)


def select_stratified_audio_files(all_audio_files, metadata_df, fraction, seed=DATASET_RANDOM_SEED):
    """Pilih subset file audio dengan seed 42 dan stratifikasi composer + instrument."""
    audio_df = pd.DataFrame({"audio_path": sorted(all_audio_files)})
    audio_df["source_file"] = audio_df["audio_path"].apply(os.path.basename)
    audio_df["track_id"] = audio_df["audio_path"].apply(_track_id_from_path)

    meta = _normalise_musicnet_metadata(metadata_df)
    if not meta.empty:
        audio_df = audio_df.merge(meta, on="track_id", how="left")

    for col in ["composer", "instrument"]:
        if col not in audio_df.columns:
            audio_df[col] = "unknown"
        audio_df[col] = audio_df[col].fillna("unknown").astype(str)

    n_files = max(1, int(round(len(audio_df) * fraction)))
    selected = stratified_sample_dataframe(
        audio_df,
        n_samples=n_files,
        stratify_cols=["composer", "instrument"],
        seed=seed,
    )
    return selected


# ============================================================
# FUNGSI PREPROCESSING
# ============================================================

def preprocess_audio(audio_path):
    """
    Preprocessing standar untuk satu file audio:
    1. Load audio
    2. Konversi ke mono
    3. Resample ke TARGET_SR (44.1 kHz)
    4. Normalisasi RMS ke target level (-23 dBFS approx)

    Menggunakan RMS normalization alih-alih peak normalization
    agar dynamic range antar segmen tetap terjaga — penting untuk
    PEAQ yang sensitif terhadap loudness statistics.
    """
    audio, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    rms = np.sqrt(np.mean(audio ** 2))
    target_rms = 0.07  # approx -23 dBFS
    if rms > 1e-6:
        audio = audio * (target_rms / rms)
    audio = np.clip(audio, -1.0, 1.0)
    return audio


def split_into_segments(audio, file_seed=0):
    """
    Potong audio panjang jadi segmen-segmen 4 detik.
    Ambil maksimal MAX_SEGMENTS_PER_FILE secara random
    agar dataset lebih beragam.

    file_seed: per-file seed agar hasil reproducible di setiap run.
    """
    rng = random.Random(file_seed)
    if len(audio) < SEGMENT_SAMPLES:
        return []
    possible_starts = list(range(0, len(audio) - SEGMENT_SAMPLES + 1, SEGMENT_SAMPLES))
    n_segments = min(MAX_SEGMENTS_PER_FILE, len(possible_starts))
    selected_starts = rng.sample(possible_starts, n_segments)
    return [audio[s : s + SEGMENT_SAMPLES] for s in selected_starts]


def apply_gap_mask(audio_segment, gap_ms, sr=TARGET_SR):
    """
    Buat versi audio dengan gap di tengah segmen.

    Gap ditempatkan di tengah agar model punya konteks
    yang seimbang di kiri dan kanan.

    Menggunakan int(round(...)) agar gap_samples selalu konsisten
    antara preprocessing dan inference — menghindari off-by-one.

    Returns:
        masked_audio : audio dengan gap diisi nol
        mask         : boolean array (True = posisi gap)
        gap_start    : indeks awal gap
        gap_end      : indeks akhir gap
    """
    gap_samples = int(round(sr * gap_ms / 1000))
    center = len(audio_segment) // 2
    gap_start = center - gap_samples // 2
    gap_end = gap_start + gap_samples

    masked_audio = audio_segment.copy()
    masked_audio[gap_start:gap_end] = 0.0

    mask = np.zeros(len(audio_segment), dtype=bool)
    mask[gap_start:gap_end] = True

    return masked_audio, mask, gap_start, gap_end


# ============================================================
# JALANKAN PREPROCESSING
# ============================================================

preprocessed_flag = os.path.join(PATHS["preprocessed"], ".done")

if SKIP_IF_EXISTS and os.path.exists(preprocessed_flag):
    print("✅ Preprocessing sudah selesai sebelumnya, skip.")
    print("   Set SKIP_IF_EXISTS = False untuk memaksa preprocessing ulang.")
else:
    audio_dir = download_musicnet()
    metadata_df = download_musicnet_metadata()

    all_audio_files = []
    for root, dirs, files in os.walk(audio_dir):
        for f in files:
            if f.endswith('.wav') or f.endswith('.flac'):
                all_audio_files.append(os.path.join(root, f))

    selected_table = select_stratified_audio_files(
        all_audio_files,
        metadata_df=metadata_df,
        fraction=DATASET_FRACTION,
        seed=DATASET_RANDOM_SEED,
    )

    print(f"\n📊 Total file audio: {len(all_audio_files)}")
    print(f"📊 File yang dipakai ({DATASET_FRACTION:.0%}): {len(selected_table)}")
    print(f"📊 Sample rate target: {TARGET_SR} Hz")
    print(f"📊 Random seed: {DATASET_RANDOM_SEED} | Stratified by composer + instrument")
    print(f"📊 Gap durations: {GAP_DURATIONS_MS} ms")
    if {"composer", "instrument"}.issubset(selected_table.columns):
        print("\n📊 Distribusi strata terpilih (top 10):")
        print(selected_table.groupby(["composer", "instrument"]).size().sort_values(ascending=False).head(10).to_string())

    segment_metadata = []
    segment_id = 0

    print("\n🔄 Memulai preprocessing...")
    for file_idx, (_, file_row) in enumerate(tqdm(selected_table.iterrows(), total=len(selected_table), desc="Preprocessing files")):
        filepath = file_row["audio_path"]
        try:
            audio = preprocess_audio(filepath)
            segments = split_into_segments(audio, file_seed=DATASET_RANDOM_SEED + file_idx)

            for segment in segments:
                clean_filename = f"seg_{segment_id:05d}.wav"
                clean_path = os.path.join(PATHS["preprocessed"], clean_filename)
                sf.write(clean_path, segment, TARGET_SR)

                for gap_ms in GAP_DURATIONS_MS:
                    masked_audio, mask, gap_start, gap_end = apply_gap_mask(segment, gap_ms)
                    masked_dir = os.path.join(PATHS["masked"], f"gap_{gap_ms}ms")
                    os.makedirs(masked_dir, exist_ok=True)
                    sf.write(os.path.join(masked_dir, clean_filename), masked_audio, TARGET_SR)

                composer = str(file_row.get("composer", "unknown"))
                instrument = str(file_row.get("instrument", "unknown"))
                segment_metadata.append({
                    "segment_id": segment_id,
                    "source_file": os.path.basename(filepath),
                    "track_id": file_row.get("track_id", _track_id_from_path(filepath)),
                    "composer": composer,
                    "instrument": instrument,
                    "stratify_key": f"{composer}__{instrument}",
                    "clean_path": clean_path,
                    "duration_s": SEGMENT_DURATION,
                    "sample_rate": TARGET_SR,
                    "n_samples": len(segment),
                })
                segment_id += 1

        except Exception as e:
            print(f"\n⚠️ Gagal memproses {filepath}: {e}")
            continue

    meta_df = pd.DataFrame(segment_metadata)
    meta_path = os.path.join(PATHS["preprocessed"], "metadata.csv")
    meta_df.to_csv(meta_path, index=False)

    with open(preprocessed_flag, 'w') as f:
        f.write("done")

    print(f"\n✅ Preprocessing selesai! Total segmen: {segment_id}")
    print(f"   Metadata: {meta_path}")

---
## CELL 4 — Definisi FiLM Layer

In [ ]:
# ============================================================
# CELL 4: DEFINISI FILM LAYER
# ============================================================
# FiLM (Feature-wise Linear Modulation) adalah jembatan antara
# encoder SSL dan decoder diffusion.
#
# Cara kerja:
#   output = gamma * fitur_decoder + beta
#   gamma dan beta dihasilkan dari latent encoder
#
# CATATAN: FiLM hanya digunakan oleh Cell 8-11 (kombinasi hybrid).
# Cell 7 (baseline) tidak menggunakan FiLM sama sekali.
# ============================================================

import torch
import torch.nn as nn


class FiLMLayer(nn.Module):
    """
    Feature-wise Linear Modulation Layer.

    Menyuntikkan informasi encoder ke dalam decoder
    dengan memodulasi fitur-fitur internal decoder.

    Support 2D (B, D) dan 3D (B, T, D) decoder features.
    Untuk 3D, gamma/beta di-broadcast ke semua timestep.

    Init: gamma=1, beta=0 (identity transform) tapi gradien non-zero.
    """

    def __init__(self, encoder_dim: int, decoder_feature_dim: int, hidden_dim: int = None):
        super(FiLMLayer, self).__init__()

        if hidden_dim is None:
            hidden_dim = (encoder_dim + decoder_feature_dim) // 2

        self.decoder_feature_dim = decoder_feature_dim

        self.proj = nn.Sequential(
            nn.Linear(encoder_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 2 * decoder_feature_dim),
        )

        nn.init.zeros_(self.proj[-1].weight)
        with torch.no_grad():
            self.proj[-1].bias[:decoder_feature_dim].fill_(1.0)
            self.proj[-1].bias[decoder_feature_dim:].fill_(0.0)

    def forward(self, encoder_latent: torch.Tensor, decoder_features: torch.Tensor):
        """
        Args:
            encoder_latent  : (B, encoder_dim)
            decoder_features: (B, D) atau (B, T, D) — last dim = decoder_feature_dim
        Returns:
            Fitur decoder yang sudah dimodulasi, shape sama dengan input
        """
        # gamma, beta: (B, decoder_feature_dim)
        gamma, beta = self.proj(encoder_latent).chunk(2, dim=-1)

        # Broadcast ke sequence dimension kalau decoder_features 3D
        if decoder_features.dim() == 3:
            gamma = gamma.unsqueeze(1)  # (B, 1, D)
            beta = beta.unsqueeze(1)    # (B, 1, D)

        return gamma * decoder_features + beta


# Konfigurasi dimensi FiLM per kombinasi
# encoder_dim        : ukuran output latent encoder
# decoder_feature_dim: ukuran fitur internal decoder yang dimodulasi
FILM_CONFIGS = {
    "clap_cqtdiff":     {"encoder_dim": 512, "decoder_feature_dim": 256},
    "clap_maid":        {"encoder_dim": 512, "decoder_feature_dim": 512},
    "audiomae_cqtdiff": {"encoder_dim": 768, "decoder_feature_dim": 256},
    "audiomae_maid":    {"encoder_dim": 768, "decoder_feature_dim": 512},
    # baseline_cqtdiff tidak ada di sini karena tidak pakai FiLM
}

print("✅ FiLMLayer berhasil didefinisikan!")
print("\n📋 Konfigurasi FiLM per kombinasi:")
for combo, cfg in FILM_CONFIGS.items():
    print(f"   {combo}: encoder_dim={cfg['encoder_dim']}, "
          f"decoder_feature_dim={cfg['decoder_feature_dim']}")
print("   baseline_cqtdiff: tidak menggunakan FiLM")

---
## CELL 5 — Fungsi Evaluasi (LSD, FAD, PEAQ_ODG)

In [ ]:
# ============================================================
# CELL 5: FUNGSI EVALUASI
# ============================================================
# Mendefinisikan 3 metrik evaluasi:
#
# 1. LSD (Log Spectral Distance)
#    - Mengukur perbedaan spektral antara audio asli vs rekonstruksi
#    - Lebih rendah = lebih baik
#    - Range: 0 (sempurna) hingga ~5 (buruk)
#
# 2. FAD (Frechet Audio Distance)
#    - Mengukur jarak distribusi audio asli vs rekonstruksi
#    - Lebih rendah = lebih baik
#    - Dihitung per set, bukan per sample
#
# 3. PEAQ_ODG (Perceptual Evaluation of Audio Quality)
#    - Mengukur kualitas perseptual berdasarkan reference vs degraded audio
#    - Skala ODG-like: 0 (imperceptible) hingga -4 (sangat buruk)
#    - Lebih tinggi (mendekati 0) = lebih baik
# ============================================================

import numpy as np
import librosa
import pandas as pd
from scipy.linalg import sqrtm

_PEAQ_FALLBACK_WARNED = False


def compute_lsd(original: np.ndarray, reconstructed: np.ndarray,
                sr: int = TARGET_SR, n_fft: int = 2048, hop_length: int = 512,
                gap_start: int = None, gap_end: int = None, frame_pad: int = 2):
    """
    Hitung Log Spectral Distance (LSD) dalam dB.

    LSD dihitung hanya pada gap region (+ frame_pad frame di tiap sisi)
    agar metrik benar-benar mengukur kualitas inpainting, bukan bagian
    non-gap yang sudah diketahui.
    """
    n = min(len(original), len(reconstructed))
    o, r = original[:n], reconstructed[:n]

    O = np.abs(librosa.stft(o, n_fft=n_fft, hop_length=hop_length)) ** 2
    R = np.abs(librosa.stft(r, n_fft=n_fft, hop_length=hop_length)) ** 2

    eps = max(1e-10, 1e-6 * O.max())
    log_diff = 10.0 * (np.log10(O + eps) - np.log10(R + eps))  # dB

    if gap_start is not None and gap_end is not None:
        f_start = max(0, gap_start // hop_length - frame_pad)
        f_end = min(O.shape[1], gap_end // hop_length + frame_pad + 1)
        log_diff = log_diff[:, f_start:f_end]

    lsd = np.mean(np.sqrt(np.mean(log_diff ** 2, axis=0)))
    return float(lsd)


def extract_fad_features(audio_list: list, sr: int = TARGET_SR):
    """
    Ekstrak fitur untuk Frechet Audio Distance.

    Primary path menggunakan VGGish embeddings (legacy FAD pipeline).
    Jika VGGish tidak tersedia di runtime, fallback ke mean log-mel
    statistics 128-dimensi agar evaluasi tetap bisa berjalan.
    """
    features = []
    try:
        import torch as _torch
        import torchvggish

        _device = _torch.device("cuda" if _torch.cuda.is_available() else "cpu")
        _vggish = torchvggish.vggish().to(_device).eval()

        with _torch.inference_mode():
            for audio in audio_list:
                audio_16k = librosa.resample(
                    np.asarray(audio, dtype=np.float32),
                    orig_sr=sr,
                    target_sr=16000,
                )
                emb = _vggish(
                    _torch.from_numpy(audio_16k).float().to(_device),
                    fs=16000,
                )
                features.append(emb.mean(0).detach().cpu().numpy())
    except Exception as exc:
        print(f"⚠️ VGGish FAD tidak tersedia/kompatibel ({exc}); fallback ke log-mel FAD features.")
        for audio in audio_list:
            mel = librosa.feature.melspectrogram(
                y=np.asarray(audio, dtype=np.float32),
                sr=sr,
                n_mels=128,
                n_fft=2048,
                hop_length=512,
            )
            mel_db = librosa.power_to_db(mel, ref=np.max)
            features.append(np.mean(mel_db, axis=1))

    return np.asarray(features, dtype=np.float64)


def compute_fad(original_audios: list, reconstructed_audios: list, sr: int = TARGET_SR):
    """
    Hitung Frechet Audio Distance (FAD) sebagai metrik distribusional per-set.

    FAD tetap terpisah dari PEAQ: FAD menjawab kemiripan distribusi embedding,
    sedangkan PEAQ_ODG menjawab kualitas perseptual per pasangan audio.
    """
    orig_features = extract_fad_features(original_audios, sr)
    recon_features = extract_fad_features(reconstructed_audios, sr)

    if orig_features.ndim == 1:
        orig_features = orig_features[:, None]
    if recon_features.ndim == 1:
        recon_features = recon_features[:, None]

    mu1 = np.mean(orig_features, axis=0)
    mu2 = np.mean(recon_features, axis=0)
    d = orig_features.shape[1]

    if len(orig_features) < 2 or len(recon_features) < 2:
        return float(np.sum((mu1 - mu2) ** 2))

    sigma1 = np.cov(orig_features, rowvar=False) + 1e-6 * np.eye(d)
    sigma2 = np.cov(recon_features, rowvar=False) + 1e-6 * np.eye(d)

    diff = mu1 - mu2
    mean_diff = np.dot(diff, diff)

    covmean, _ = sqrtm(sigma1 @ sigma2, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fad = mean_diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return float(np.real(fad))


def _coerce_metric_score(value):
    """Ambil scalar score dari return value metric yang mungkin dict/tuple/array."""
    if value is None:
        return None
    if isinstance(value, dict):
        for key in ["odg", "ODG", "peaq", "PEAQ", "score", "value"]:
            if key in value:
                return _coerce_metric_score(value[key])
        return None
    if hasattr(value, "odg"):
        return _coerce_metric_score(value.odg)
    if hasattr(value, "score"):
        return _coerce_metric_score(value.score)
    try:
        arr = np.asarray(value, dtype=float).reshape(-1)
        if arr.size > 0 and np.isfinite(arr[0]):
            return float(arr[0])
    except Exception:
        return None
    return None


def _try_call_metric(metric, original, reconstructed, sr):
    """Coba beberapa signature umum untuk fungsi/class metric audio."""
    call_attempts = [
        lambda obj: obj(original, reconstructed, sr),
        lambda obj: obj(original, reconstructed, sample_rate=sr),
        lambda obj: obj(original, reconstructed, fs=sr),
        lambda obj: obj(reference=original, degraded=reconstructed, sample_rate=sr),
        lambda obj: obj(ref=original, deg=reconstructed, fs=sr),
    ]

    for call in call_attempts:
        try:
            score = _coerce_metric_score(call(metric))
            if score is not None:
                return score
        except Exception:
            continue
    return None


def _try_aquatk_peaqb(original, reconstructed, sr):
    """
    Primary path: AquaTK Basic PEAQ (PEAQb).
    Dibuat defensif karena AquaTK masih in-development dan API dapat berubah.
    """
    import importlib
    import inspect

    candidates = [
        ("aquatk.metrics", "PEAQb"),
        ("aquatk.metrics", "PEAQ"),
        ("aquatk.metrics.peaqb", "PEAQb"),
        ("aquatk.metrics.peaq", "PEAQb"),
        ("aquatk.metrics.peaq", "PEAQ"),
        ("aquatk", "PEAQb"),
    ]

    for module_name, attr_name in candidates:
        try:
            module = importlib.import_module(module_name)
            metric_ctor = getattr(module, attr_name)
        except Exception:
            continue

        if inspect.isclass(metric_ctor):
            instances = []
            for kwargs in [{"sample_rate": sr}, {"sr": sr}, {"fs": sr}, {}]:
                try:
                    instances.append(metric_ctor(**kwargs))
                except Exception:
                    continue
            for metric in instances:
                for method_name in ["compute", "score", "evaluate", "__call__"]:
                    method = getattr(metric, method_name, None)
                    if method is None:
                        continue
                    score = _try_call_metric(method, original, reconstructed, sr)
                    if score is not None:
                        return score
        else:
            score = _try_call_metric(metric_ctor, original, reconstructed, sr)
            if score is not None:
                return score

    return None


def _compute_visqol_odg(original, reconstructed, sr):
    """Fallback perceptual ODG via ViSQOL music mode."""
    from visqol import visqol_lib_py
    from visqol.pb2 import visqol_config_pb2

    config = visqol_config_pb2.VisqolConfig()
    config.audio.sample_rate = 48000
    config.options.use_speech_scoring = False  # music mode

    api = visqol_lib_py.VisqolApi()
    api.Create(config)

    orig_48k = librosa.resample(original, orig_sr=sr, target_sr=48000).astype(np.float64)
    recon_48k = librosa.resample(reconstructed, orig_sr=sr, target_sr=48000).astype(np.float64)
    n = min(len(orig_48k), len(recon_48k))
    orig_48k, recon_48k = orig_48k[:n], recon_48k[:n]

    moslqo = api.Measure(orig_48k, recon_48k).moslqo  # 1.0 .. 5.0
    return float(moslqo - 5.0)  # Map to [-4, 0] range


def _compute_nsim_odg(original, reconstructed, sr):
    """Last-resort fallback: NSIM pada log-mel, dipetakan ke ODG-like scale."""
    orig_mel = librosa.feature.melspectrogram(
        y=original, sr=sr, n_mels=128, n_fft=2048, hop_length=512
    )
    recon_mel = librosa.feature.melspectrogram(
        y=reconstructed, sr=sr, n_mels=128, n_fft=2048, hop_length=512
    )

    orig_log = librosa.power_to_db(orig_mel, ref=np.max)
    recon_log = librosa.power_to_db(recon_mel, ref=np.max)

    C1, C2 = 1e-4, 1e-4
    mu_o = np.mean(orig_log, axis=0)
    mu_r = np.mean(recon_log, axis=0)
    sig_o = np.std(orig_log, axis=0)
    sig_r = np.std(recon_log, axis=0)
    sig_or = np.mean((orig_log - mu_o) * (recon_log - mu_r), axis=0)

    ssim = ((2 * mu_o * mu_r + C1) * (2 * sig_or + C2)) / \
           ((mu_o**2 + mu_r**2 + C1) * (sig_o**2 + sig_r**2 + C2))
    mean_ssim = float(np.mean(np.clip(ssim, 0, 1)))

    odg = -4.0 * (1.0 - mean_ssim ** 0.5)
    return float(np.clip(odg, -4.0, 0.0))


def compute_peaq_odg(original: np.ndarray, reconstructed: np.ndarray, sr: int = TARGET_SR):
    """
    Hitung PEAQ perceptual quality sebagai Objective Difference Grade-like score.

    Primary path memakai AquaTK PEAQb (Basic PEAQ) jika package tersedia.
    Jika PEAQb tidak tersedia di runtime, fallback ke ViSQOL music mode,
    lalu fallback terakhir ke NSIM log-mel agar evaluasi tetap bisa berjalan.
    """
    global _PEAQ_FALLBACK_WARNED

    min_len = min(len(original), len(reconstructed))
    original = np.asarray(original[:min_len], dtype=np.float64)
    reconstructed = np.asarray(reconstructed[:min_len], dtype=np.float64)

    # PEAQ dirancang untuk 44.1/48kHz; target pipeline sekarang 44.1kHz.
    peaq_sr = sr
    if peaq_sr not in (44100, 48000):
        original = librosa.resample(original, orig_sr=sr, target_sr=44100)
        reconstructed = librosa.resample(reconstructed, orig_sr=sr, target_sr=44100)
        peaq_sr = 44100

    try:
        score = _try_aquatk_peaqb(original, reconstructed, peaq_sr)
        if score is not None and np.isfinite(score):
            return float(np.clip(score, -4.0, 0.0))
    except Exception:
        pass

    if not _PEAQ_FALLBACK_WARNED:
        print("⚠️ AquaTK PEAQb tidak tersedia/kompatibel; fallback ke ViSQOL/NSIM untuk PEAQ_ODG.")
        _PEAQ_FALLBACK_WARNED = True

    try:
        return _compute_visqol_odg(original, reconstructed, peaq_sr)
    except Exception:
        return _compute_nsim_odg(original, reconstructed, peaq_sr)


# Backward-compatible aliases.
compute_peaq = compute_peaq_odg
compute_odg = compute_peaq_odg


def evaluate_all_gaps(original_audios: list, reconstructed_dict: dict, sr: int = TARGET_SR):
    """
    Evaluasi semua gap duration untuk satu model.

    Returns:
        DataFrame dengan kolom `gap_ms`, `LSD`, `FAD`, dan `PEAQ_ODG`.
    """
    results = []
    for gap_ms, recon_audios in reconstructed_dict.items():
        print(f"  Evaluating gap {gap_ms}ms...")

        # Hitung gap indices (konsisten dengan apply_gap_mask)
        gap_samples = int(round(sr * gap_ms / 1000))

        lsd_scores = []
        peaq_odg_scores = []
        for orig, recon in zip(original_audios, recon_audios):
            center = len(orig) // 2
            gap_start = center - gap_samples // 2
            gap_end = gap_start + gap_samples

            lsd_scores.append(compute_lsd(orig, recon, sr,
                                          gap_start=gap_start, gap_end=gap_end))
            peaq_odg_scores.append(compute_peaq_odg(orig, recon, sr))

        fad_score = compute_fad(original_audios, recon_audios, sr)

        results.append({
            "gap_ms": gap_ms,
            "LSD": round(np.mean(lsd_scores), 4),
            "FAD": round(fad_score, 4),
            "PEAQ_ODG": round(np.mean(peaq_odg_scores), 4),
        })

    return pd.DataFrame(results)


print("✅ Fungsi evaluasi (LSD, FAD, PEAQ_ODG) berhasil didefinisikan!")

---
## CELL 6 — Helper Functions

In [ ]:
# ============================================================
# CELL 6: HELPER FUNCTIONS
# ============================================================
# Fungsi pendukung untuk:
# - Monitoring penggunaan VRAM
# - Membersihkan memori GPU setelah selesai satu model
# - Menyimpan hasil ke Google Drive
# - Mengecek apakah model sudah pernah dijalankan
# - Loading data yang sudah dipreprocess
# ============================================================

import torch
import gc
import os
import pandas as pd
import soundfile as sf
import random
from datetime import datetime


def print_gpu_usage(label: str = ""):
    """
    Tampilkan penggunaan VRAM saat ini.

    Cara pakai:
        print_gpu_usage("Sebelum load encoder")
        # load model...
        print_gpu_usage("Sesudah load encoder")
    """
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved() / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        free      = total - reserved
        label_str = f"[{label}] " if label else ""
        print(f"🖥️  GPU {label_str}| Terpakai: {allocated:.2f}GB | "
              f"Reserved: {reserved:.2f}GB | Bebas: {free:.2f}GB / {total:.2f}GB")


def clear_gpu_memory(*models):
    """
    Bebaskan VRAM setelah selesai menggunakan model.

    Cara pakai:
        clear_gpu_memory(encoder, decoder, film_layer)
        # atau untuk baseline:
        clear_gpu_memory(decoder)
    """
    print_gpu_usage("Sebelum clear")
    for model in models:
        if model is not None:
            del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print_gpu_usage("Sesudah clear")
    print("✅ GPU memory berhasil dibersihkan\n")


def save_results(results_df: pd.DataFrame, model_name: str):
    """
    Simpan hasil evaluasi ke Google Drive.

    Menyimpan dua file:
    1. File per model: {model_name}_results.csv
    2. Master file: all_results.csv (gabungan semua model)

    Args:
        results_df : DataFrame hasil evaluasi
        model_name : Nama model (misal: "baseline_cqtdiff", "clap_cqtdiff")
    """
    results_df = results_df.copy()
    results_df["model"] = model_name
    results_df["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Simpan file per model
    model_path = os.path.join(PATHS["results"], f"{model_name}_results.csv")
    results_df.to_csv(model_path, index=False)
    print(f"💾 Hasil {model_name} disimpan: {model_path}")

    # Update master file
    master_path = os.path.join(PATHS["results"], "all_results.csv")
    if os.path.exists(master_path):
        existing = pd.read_csv(master_path)
        existing = existing[existing["model"] != model_name]  # Hapus hasil lama
        combined = pd.concat([existing, results_df], ignore_index=True)
    else:
        combined = results_df

    combined.to_csv(master_path, index=False)
    print(f"💾 Master file diupdate: {master_path}")


def check_if_done(model_name: str):
    """
    Cek apakah model ini sudah pernah dijalankan.

    Berguna saat Colab crash: model yang sudah selesai
    tidak perlu diulang.

    Returns:
        True  : sudah selesai, bisa di-skip
        False : belum selesai, perlu dijalankan
    """
    result_path = os.path.join(PATHS["results"], f"{model_name}_results.csv")
    if os.path.exists(result_path):
        print(f"✅ {model_name} sudah selesai sebelumnya.")
        print(f"   Untuk menjalankan ulang, hapus: {result_path}")
        return True
    return False


def _stratified_sample_table(df, n_samples, seed=DATASET_RANDOM_SEED,
                             stratify_cols=("composer", "instrument")):
    """Sampling deterministic dengan proporsi composer + instrument sebisa mungkin terjaga."""
    if len(df) == 0 or n_samples <= 0:
        return df.iloc[0:0].copy()

    work = df.copy().reset_index(drop=True)
    work["_sample_row_id"] = np.arange(len(work))
    n_samples = min(int(n_samples), len(work))
    for col in stratify_cols:
        if col not in work.columns:
            work[col] = "unknown"
        work[col] = work[col].fillna("unknown").astype(str)

    rng = np.random.default_rng(seed)
    grouped = list(work.groupby(list(stratify_cols), dropna=False, sort=True))
    quotas = []
    for _, group in grouped:
        expected = len(group) * n_samples / len(work)
        base = int(np.floor(expected))
        quotas.append({
            "group": group,
            "quota": min(base, len(group)),
            "fractional": expected - base,
            "tie": rng.random(),
        })

    remaining = n_samples - sum(q["quota"] for q in quotas)
    for q in sorted(quotas, key=lambda item: (-item["fractional"], item["tie"])):
        if remaining <= 0:
            break
        capacity = len(q["group"]) - q["quota"]
        if capacity > 0:
            q["quota"] += 1
            remaining -= 1

    parts = []
    for offset, q in enumerate(quotas):
        if q["quota"] > 0:
            parts.append(q["group"].sample(q["quota"], random_state=seed + offset))

    sampled = pd.concat(parts, ignore_index=True) if parts else work.iloc[0:0].copy()
    if len(sampled) < n_samples:
        missing = n_samples - len(sampled)
        sampled_ids = set(sampled["_sample_row_id"]) if "_sample_row_id" in sampled.columns else set()
        unsampled = work[~work["_sample_row_id"].isin(sampled_ids)]
        if len(unsampled) > 0:
            sampled = pd.concat([
                sampled,
                unsampled.sample(min(missing, len(unsampled)), random_state=seed + 999)
            ], ignore_index=True)

    return sampled.sample(frac=1.0, random_state=seed).drop(columns=["_sample_row_id"], errors="ignore").reset_index(drop=True)


def get_data_splits(meta_df=None):
    """
    Buat group-aware train/val/test split berdasarkan source_file.

    Split dilakukan pada level source_file (lagu), bukan segment,
    agar tidak ada segment dari lagu yang sama muncul di train dan test
    (mencegah data leakage). Pemilihan source_file memakai seed 42 dan
    stratified sampling berdasarkan composer + instrument.

    Returns:
        dict: {"train": DataFrame, "val": DataFrame, "test": DataFrame}
    """
    if meta_df is None:
        meta_path = os.path.join(PATHS["preprocessed"], "metadata.csv")
        meta_df = pd.read_csv(meta_path)

    source_cols = ["source_file"]
    for col in ["composer", "instrument"]:
        if col in meta_df.columns:
            source_cols.append(col)

    source_df = meta_df[source_cols].drop_duplicates("source_file").reset_index(drop=True)
    for col in ["composer", "instrument"]:
        if col not in source_df.columns:
            source_df[col] = "unknown"
        source_df[col] = source_df[col].fillna("unknown").astype(str)

    n_test = max(1, int(round(len(source_df) * 0.15)))
    n_val = max(1, int(round(len(source_df) * 0.15)))

    test_sources_df = _stratified_sample_table(source_df, n_test, seed=DATASET_RANDOM_SEED)
    remaining_sources = source_df[~source_df["source_file"].isin(test_sources_df["source_file"])].reset_index(drop=True)
    val_sources_df = _stratified_sample_table(remaining_sources, n_val, seed=DATASET_RANDOM_SEED + 1)
    train_sources_df = remaining_sources[~remaining_sources["source_file"].isin(val_sources_df["source_file"])].reset_index(drop=True)

    test_files = set(test_sources_df["source_file"])
    val_files = set(val_sources_df["source_file"])
    train_files = set(train_sources_df["source_file"])

    splits = {
        "train": meta_df[meta_df["source_file"].isin(train_files)].reset_index(drop=True),
        "val": meta_df[meta_df["source_file"].isin(val_files)].reset_index(drop=True),
        "test": meta_df[meta_df["source_file"].isin(test_files)].reset_index(drop=True),
    }

    for name, df in splits.items():
        n_sources = df["source_file"].nunique()
        print(f"  {name:5s}: {len(df)} segments from {n_sources} source files")

    print(f"  split seed: {DATASET_RANDOM_SEED} | stratified by composer + instrument")
    return splits

def load_preprocessed_data(n_samples: int = 50, split: str = "test"):
    """
    Load data yang sudah dipreprocess dari Drive.

    Menggunakan group-aware split untuk mencegah data leakage.
    Default menggunakan split "test" untuk evaluasi.

    Args:
        n_samples: Jumlah sampel untuk evaluasi.
        split: "train", "val", atau "test"

    Returns:
        original_audios : List audio ground truth
        masked_by_gap   : {gap_ms: [list audio masked]}
    """
    meta_path = os.path.join(PATHS["preprocessed"], "metadata.csv")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(
            "Metadata tidak ditemukan! Jalankan Cell 3 (preprocessing) terlebih dahulu."
        )

    meta_df = pd.read_csv(meta_path)
    splits = get_data_splits(meta_df)
    split_df = splits[split]

    selected = _stratified_sample_table(split_df, min(n_samples, len(split_df)), seed=DATASET_RANDOM_SEED + 2)

    print(f"📂 Loading {len(selected)} sampel dari Drive (split={split})...")

    original_audios = []
    masked_by_gap = {gap_ms: [] for gap_ms in GAP_DURATIONS_MS}

    for _, row in selected.iterrows():
        orig_audio, _ = sf.read(row["clean_path"])
        original_audios.append(orig_audio)

        filename = os.path.basename(row["clean_path"])
        for gap_ms in GAP_DURATIONS_MS:
            masked_path = os.path.join(PATHS["masked"], f"gap_{gap_ms}ms", filename)
            masked_audio, _ = sf.read(masked_path)
            masked_by_gap[gap_ms].append(masked_audio)

    print(f"✅ {len(original_audios)} sampel siap dievaluasi.")
    return original_audios, masked_by_gap


print("✅ Helper functions berhasil didefinisikan!")

---
## CELL 6.5 — Dataset, DataLoader & Shared Utilities

Definisi Dataset/DataLoader untuk training, serta fungsi utilitas
yang dipakai di semua cell inference (mask creation, boundary cross-fade).

In [ ]:
# ============================================================
# CELL 6.5: DATASET, DATALOADER & SHARED UTILITIES
# ============================================================
# 1. MusicGapDataset: PyTorch Dataset untuk training
# 2. make_gap_mask(): fungsi mask yang konsisten antara
#    preprocessing dan inference (menghindari off-by-one)
# 3. crossfade_boundary(): half-cosine crossfade untuk
#    menghilangkan click artifact di boundary gap
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import soundfile as sf
import os


def make_gap_mask(audio_length, gap_ms, sr=TARGET_SR):
    """
    Buat gap mask yang KONSISTEN dengan apply_gap_mask().

    Fungsi ini dipakai di semua cell inference (Cell 7-11)
    agar mask selalu identik dengan yang dibuat saat preprocessing.
    Menghindari bug off-by-one dari pendekatan sebelumnya
    (center - gap_samples//2 : center + gap_samples//2).
    """
    gap_samples = int(round(sr * gap_ms / 1000))
    center = audio_length // 2
    gap_start = center - gap_samples // 2
    gap_end = gap_start + gap_samples

    mask = np.zeros(audio_length, dtype=bool)
    mask[gap_start:gap_end] = True
    return mask, gap_start, gap_end


def crossfade_boundary(original, reconstructed, gap_start, gap_end,
                       sr=TARGET_SR, fade_ms=30):
    """
    Terapkan half-cosine crossfade di boundary gap untuk
    menghilangkan click artifact.

    Di boundary kiri (gap_start): fade dari original ke reconstructed
    Di boundary kanan (gap_end): fade dari reconstructed ke original
    """
    fade_n = int(fade_ms * 1e-3 * sr)
    if fade_n <= 0:
        return reconstructed

    output = reconstructed.copy()
    fade = 0.5 * (1 - np.cos(np.pi * np.linspace(0, 1, fade_n)))

    # Left boundary
    left_start = max(0, gap_start - fade_n)
    left_len = gap_start - left_start
    if left_len > 0:
        f = fade[-left_len:]
        output[left_start:gap_start] = (
            original[left_start:gap_start] * (1 - f)
            + reconstructed[left_start:gap_start] * f
        )

    # Right boundary
    right_end = min(len(output), gap_end + fade_n)
    right_len = right_end - gap_end
    if right_len > 0:
        f = fade[:right_len]
        output[gap_end:right_end] = (
            reconstructed[gap_end:right_end] * (1 - f)
            + original[gap_end:right_end] * f
        )

    return output


class MusicGapDataset(Dataset):
    """
    PyTorch Dataset untuk training music inpainting.

    Setiap __getitem__ menghasilkan:
    - clean: audio asli (4 detik)
    - masked: audio dengan gap nol
    - mask: boolean mask (True = gap region)
    - gap_start, gap_end: indeks gap
    - gap_ms: durasi gap dalam ms
    """
    def __init__(self, meta_df, gap_ms_choices, sr=TARGET_SR):
        self.meta = meta_df.reset_index(drop=True)
        self.gap_ms_choices = gap_ms_choices
        self.sr = sr

    def __len__(self):
        return len(self.meta) * len(self.gap_ms_choices)

    def __getitem__(self, idx):
        seg_i, g_i = divmod(idx, len(self.gap_ms_choices))
        row = self.meta.iloc[seg_i]
        gap_ms = self.gap_ms_choices[g_i]

        clean, _ = sf.read(row["clean_path"])
        masked, mask, gs, ge = apply_gap_mask(clean, gap_ms, self.sr)

        return {
            "clean": torch.from_numpy(clean).float(),
            "masked": torch.from_numpy(masked).float(),
            "mask": torch.from_numpy(mask),
            "gap_start": gs,
            "gap_end": ge,
            "gap_ms": gap_ms,
        }


def make_dataloaders(batch_size=16, num_workers=2):
    """
    Buat DataLoader untuk train/val/test dengan group-aware split.
    """
    meta_path = os.path.join(PATHS["preprocessed"], "metadata.csv")
    meta_df = pd.read_csv(meta_path)
    splits = get_data_splits(meta_df)

    loaders = {}
    for name, df in splits.items():
        ds = MusicGapDataset(df, GAP_DURATIONS_MS, sr=TARGET_SR)
        loaders[name] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(name == "train"),
            num_workers=num_workers,
            pin_memory=True,
            persistent_workers=(num_workers > 0),
        )
    return loaders


print("✅ Dataset, DataLoader & shared utilities berhasil didefinisikan!")
print(f"   Gap durations: {GAP_DURATIONS_MS} ms")

---
## CELL 6.6 — Training Loop (Reconstruction-Based)

**Perubahan penting dari versi sebelumnya:**
Versi lama memakai DDPM epsilon prediction untuk training, tapi saat inference
langsung pakai output model sebagai audio. Ini menyebabkan **mismatch fatal**:
model dilatih memprediksi noise, tapi output-nya dipakai sebagai rekonstruksi.

Versi baru ini memakai **reconstruction loss di STFT domain**:
- Training: prediksi STFT magnitude clean audio, loss pada gap frames
- Inference: prediksi STFT magnitude → iSTFT → replace gap region
- Training dan inference **fully aligned**

Fitur:
- Gap-only reconstruction loss (L1 pada STFT magnitude di gap region)
- Full audio auxiliary loss (0.1x bobot, buat stabilitas)
- Classifier-free guidance dropout (CFG)
- Mixed precision training (AMP) + gradient clipping
- Separate trainer untuk baseline (tanpa encoder) dan hybrid (dengan encoder)

In [ ]:
# ============================================================
# CELL 6.6: TRAINING LOOP (RECONSTRUCTION-BASED)
# ============================================================
# Training untuk pipeline hybrid SSL + Decoder.
#
# PERBAIKAN UTAMA dari versi sebelumnya:
# - Ganti epsilon prediction (DDPM) -> reconstruction loss
#   karena proxy model terlalu simpel untuk proper DDPM
#   (butuh UNet + timestep embedding + iterative sampling)
# - Training dan inference sekarang ALIGNED:
#   training prediksi STFT magnitude clean, inference juga
# - Loss dihitung pada gap region only (spectral domain)
# - CFG dropout tetap dipertahankan
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import os
from tqdm import tqdm


def compute_stft_target(clean_audio, n_fft=2048, hop_length=512):
    """Hitung STFT magnitude target dari clean audio."""
    spec = torch.stft(
        clean_audio, n_fft=n_fft, hop_length=hop_length,
        return_complex=True
    )
    return spec.abs()  # (B, F, T)


def compute_frame_mask(sample_mask, n_frames, hop_length=512):
    """Konversi sample-level mask -> frame-level mask buat loss."""
    B = sample_mask.shape[0]
    device = sample_mask.device
    frame_mask = torch.zeros(B, n_frames, device=device)
    for i in range(n_frames):
        start = i * hop_length
        end = min(start + hop_length, sample_mask.shape[-1])
        frame_mask[:, i] = sample_mask[:, start:end].float().mean(dim=-1)
    return frame_mask > 0.5  # (B, T) boolean


def train_step_reconstruction(decoder, encoder_fn, film, batch, optimizer, scaler,
                              cfg_drop=0.1, device="cuda"):
    """
    Satu step training dengan reconstruction loss di STFT domain.

    Aligned dengan inference: model prediksi STFT magnitude,
    loss dihitung pada gap frames only.
    """
    decoder.train()
    film.train()

    clean = batch["clean"].to(device, non_blocking=True)
    masked = batch["masked"].to(device, non_blocking=True)
    mask = batch["mask"].to(device, non_blocking=True)
    B = clean.size(0)

    # Encoder latent (frozen, no grad)
    with torch.no_grad():
        z = encoder_fn(masked)

    # CFG dropout: randomly zero out conditioning
    drop = (torch.rand(B, device=device) < cfg_drop).view(-1, *([1] * (z.dim() - 1)))
    z = torch.where(drop, torch.zeros_like(z), z)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=scaler.is_enabled()):
        # Extract features dari masked audio (dengan mask info)
        features = decoder.get_features(masked, mask)  # (B, T, feature_dim)

        # Inject FiLM conditioning dari encoder
        cond_features = film(z.float(), features)  # (B, T, feature_dim)

        # Decode ke STFT magnitude prediction
        pred_mag = decoder.decode_features(cond_features)  # (B, T, F)

        # Target: STFT magnitude dari clean audio
        target_mag = compute_stft_target(clean).permute(0, 2, 1)  # (B, T, F)

        # Sesuaikan ukuran temporal
        T_min = min(pred_mag.shape[1], target_mag.shape[1])
        pred_mag = pred_mag[:, :T_min, :]
        target_mag = target_mag[:, :T_min, :]

        # Frame mask untuk gap-only loss
        frame_mask = compute_frame_mask(mask, T_min).unsqueeze(-1)  # (B, T, 1)
        frame_mask = frame_mask.expand_as(pred_mag)

        # Loss: L1 pada gap region + 0.1 * L1 full (buat stabilitas)
        gap_loss = F.l1_loss(pred_mag[frame_mask], target_mag[frame_mask])
        full_loss = F.l1_loss(pred_mag, target_mag)
        loss = gap_loss + 0.1 * full_loss

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        list(decoder.parameters()) + list(film.parameters()), max_norm=1.0
    )
    scaler.step(optimizer)
    scaler.update()

    return {
        "loss": loss.item(),
        "gap_loss": gap_loss.item(),
        "full_loss": full_loss.item(),
    }


def train_step_baseline(decoder, batch, optimizer, scaler, device="cuda"):
    """
    Training step buat baseline (tanpa encoder, tanpa FiLM).
    Sama kayak train_step_reconstruction tapi tanpa conditioning.
    """
    decoder.train()

    clean = batch["clean"].to(device, non_blocking=True)
    masked = batch["masked"].to(device, non_blocking=True)
    mask = batch["mask"].to(device, non_blocking=True)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=scaler.is_enabled()):
        features = decoder.get_features(masked, mask)
        pred_mag = decoder.decode_features(features)  # (B, T, F)

        target_mag = compute_stft_target(clean).permute(0, 2, 1)  # (B, T, F)

        T_min = min(pred_mag.shape[1], target_mag.shape[1])
        pred_mag = pred_mag[:, :T_min, :]
        target_mag = target_mag[:, :T_min, :]

        frame_mask = compute_frame_mask(mask, T_min).unsqueeze(-1)
        frame_mask = frame_mask.expand_as(pred_mag)

        gap_loss = F.l1_loss(pred_mag[frame_mask], target_mag[frame_mask])
        full_loss = F.l1_loss(pred_mag, target_mag)
        loss = gap_loss + 0.1 * full_loss

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()

    return {"loss": loss.item(), "gap_loss": gap_loss.item()}


def train_model(decoder, encoder_fn, film, train_loader, val_loader=None,
                num_epochs=50, lr=1e-4, device="cuda", checkpoint_dir=None,
                model_name="model"):
    """
    Training loop lengkap untuk hybrid model (encoder + FiLM + decoder).
    Pakai reconstruction loss di STFT domain.
    """
    optimizer = torch.optim.AdamW(
        list(decoder.parameters()) + list(film.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        decoder.train()
        film.train()
        epoch_losses = []

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            metrics = train_step_reconstruction(
                decoder, encoder_fn, film, batch, optimizer, scaler, device=device
            )
            epoch_losses.append(metrics["loss"])

        avg_loss = np.mean(epoch_losses)
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.6f} | LR: {lr_now:.2e}")
        scheduler.step()

        # Validasi tiap 5 epoch
        if val_loader is not None and (epoch + 1) % 5 == 0:
            val_loss = validate_model(decoder, encoder_fn, film, val_loader, device)
            print(f"  Val Loss: {val_loss:.6f}")

            if checkpoint_dir and val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(decoder, film, optimizer, epoch, val_loss,
                                checkpoint_dir, model_name)

    # Fallback: simpan checkpoint terakhir kalau belum pernah tersimpan
    if checkpoint_dir and best_val_loss == float("inf"):
        save_checkpoint(decoder, film, optimizer, num_epochs - 1, avg_loss,
                        checkpoint_dir, model_name)

    print(f"\n✅ Training selesai! Best val loss: {best_val_loss:.6f}")
    return decoder, film


def train_baseline_model(decoder, train_loader, val_loader=None,
                         num_epochs=50, lr=1e-4, device="cuda",
                         checkpoint_dir=None, model_name="baseline_cqtdiff"):
    """
    Training loop buat baseline (tanpa encoder, tanpa FiLM).
    Biar perbandingan fair: baseline juga di-train, bedanya cuma tanpa conditioning.
    """
    optimizer = torch.optim.AdamW(decoder.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        decoder.train()
        epoch_losses = []

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            metrics = train_step_baseline(decoder, batch, optimizer, scaler, device=device)
            epoch_losses.append(metrics["loss"])

        avg_loss = np.mean(epoch_losses)
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.6f} | LR: {lr_now:.2e}")
        scheduler.step()

        # Validasi
        if val_loader is not None and (epoch + 1) % 5 == 0:
            val_loss = validate_baseline(decoder, val_loader, device)
            print(f"  Val Loss: {val_loss:.6f}")

            if checkpoint_dir and val_loss < best_val_loss:
                best_val_loss = val_loss
                os.makedirs(checkpoint_dir, exist_ok=True)
                ckpt_path = os.path.join(checkpoint_dir, f"{model_name}_best.pt")
                torch.save({
                    "epoch": epoch,
                    "decoder_state": decoder.state_dict(),
                    "val_loss": val_loss,
                }, ckpt_path)
                print(f"  💾 Best checkpoint saved: {ckpt_path}")

    if checkpoint_dir and best_val_loss == float("inf"):
        os.makedirs(checkpoint_dir, exist_ok=True)
        ckpt_path = os.path.join(checkpoint_dir, f"{model_name}_best.pt")
        torch.save({
            "epoch": num_epochs - 1,
            "decoder_state": decoder.state_dict(),
            "val_loss": avg_loss,
        }, ckpt_path)

    print(f"\n✅ Training baseline selesai! Best val loss: {best_val_loss:.6f}")
    return decoder


def validate_model(decoder, encoder_fn, film, val_loader, device):
    """Validasi hybrid model (dengan encoder + FiLM)."""
    decoder.eval()
    film.eval()
    val_losses = []

    with torch.inference_mode():
        for batch in val_loader:
            clean = batch["clean"].to(device)
            masked = batch["masked"].to(device)
            mask = batch["mask"].to(device)

            z = encoder_fn(masked)
            features = decoder.get_features(masked, mask)
            cond_features = film(z.float(), features)
            pred_mag = decoder.decode_features(cond_features)

            target_mag = compute_stft_target(clean).permute(0, 2, 1)
            T_min = min(pred_mag.shape[1], target_mag.shape[1])
            pred_mag = pred_mag[:, :T_min, :]
            target_mag = target_mag[:, :T_min, :]

            frame_mask = compute_frame_mask(mask, T_min).unsqueeze(-1).expand_as(pred_mag)
            val_loss = F.l1_loss(pred_mag[frame_mask], target_mag[frame_mask])
            val_losses.append(val_loss.item())

    return float(np.mean(val_losses))


def validate_baseline(decoder, val_loader, device):
    """Validasi baseline (tanpa encoder)."""
    decoder.eval()
    val_losses = []

    with torch.inference_mode():
        for batch in val_loader:
            clean = batch["clean"].to(device)
            masked = batch["masked"].to(device)
            mask = batch["mask"].to(device)

            features = decoder.get_features(masked, mask)
            pred_mag = decoder.decode_features(features)

            target_mag = compute_stft_target(clean).permute(0, 2, 1)
            T_min = min(pred_mag.shape[1], target_mag.shape[1])
            pred_mag = pred_mag[:, :T_min, :]
            target_mag = target_mag[:, :T_min, :]

            frame_mask = compute_frame_mask(mask, T_min).unsqueeze(-1).expand_as(pred_mag)
            val_loss = F.l1_loss(pred_mag[frame_mask], target_mag[frame_mask])
            val_losses.append(val_loss.item())

    return float(np.mean(val_losses))


def save_checkpoint(decoder, film, optimizer, epoch, val_loss, checkpoint_dir, model_name):
    """Simpan checkpoint model."""
    os.makedirs(checkpoint_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_dir, f"{model_name}_best.pt")
    torch.save({
        "epoch": epoch,
        "decoder_state": decoder.state_dict(),
        "film_state": film.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_loss": val_loss,
    }, ckpt_path)
    print(f"  💾 Best checkpoint saved: {ckpt_path}")


print("✅ Training loop (reconstruction-based) berhasil didefinisikan!")
print("   Komponen: train_step_reconstruction, train_step_baseline,")
print("             train_model, train_baseline_model, validate_model")

In [ ]:
# ============================================================
# CELL 6.7: SHARED HYBRID TRAINING HELPERS
# ============================================================
# Helper bersama untuk:
# - builder encoder batch-capable (CLAP, AudioMAE)
# - builder decoder shared (CQT-Diff+ proxy, MAID replica)
# - save/load checkpoint hybrid
# - trainer minimal-kompatibel untuk MAID
# - helper evaluasi hybrid yang selalu load checkpoint terlatih
#
# PERBAIKAN UTAMA:
# - SharedCQTDiffProxyModel sekarang temporal-aware (per-frame STFT)
# - Training & inference aligned (keduanya reconstruction-based)
# - Mask-aware: model tahu lokasi dan ukuran gap
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa

HYBRID_CKPT_SUFFIX = "_best.pt"


def get_model_checkpoint_dir(model_name: str):
    ckpt_dir = os.path.join(PATHS["checkpoints"], model_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    return ckpt_dir


def get_model_checkpoint_path(model_name: str):
    return os.path.join(get_model_checkpoint_dir(model_name), f"{model_name}{HYBRID_CKPT_SUFFIX}")


def hybrid_checkpoint_exists(model_name: str):
    return os.path.exists(get_model_checkpoint_path(model_name))


def load_hybrid_checkpoint(model_name: str, decoder: nn.Module, film_layer: nn.Module, device):
    ckpt_path = get_model_checkpoint_path(model_name)
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"Checkpoint untuk {model_name} belum ada: {ckpt_path}. Jalankan cell training-nya terlebih dahulu."
        )

    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    decoder.load_state_dict(payload["decoder_state"])
    film_layer.load_state_dict(payload["film_state"])
    print(f"✅ Checkpoint diload: {ckpt_path}")
    return payload


def load_baseline_checkpoint(decoder: nn.Module, device):
    """Load checkpoint baseline (tanpa FiLM)."""
    ckpt_dir = get_model_checkpoint_dir("baseline_cqtdiff")
    ckpt_path = os.path.join(ckpt_dir, "baseline_cqtdiff_best.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Baseline checkpoint belum ada: {ckpt_path}")
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    decoder.load_state_dict(payload["decoder_state"])
    print(f"✅ Baseline checkpoint diload: {ckpt_path}")
    return payload


class SharedCQTDiffProxyModel(nn.Module):
    """
    Proxy CQT-Diff+ dengan arsitektur temporal (STFT-based).

    PERBAIKAN dari versi sebelumnya:
    1. Temporal-aware: process per-frame STFT, bukan mean pool
    2. Mask-aware: model tahu dimana gap-nya lewat mask channel
    3. Training-inference aligned: keduanya prediksi STFT magnitude
    4. Proper reconstruction via iSTFT (bukan MLP -> full waveform)

    Arsitektur:
    - Encoder: STFT mag + mask -> per-frame features (B, T, feature_dim)
    - Temporal: Transformer buat konteks temporal antar frame
    - Decoder: features -> predicted STFT magnitude (B, T, F)
    - Inpaint: predicted mag + phase dari input -> iSTFT -> replace gap
    """

    def __init__(self, n_fft=2048, hop_length=512, feature_dim=256):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.feature_dim = feature_dim
        self.freq_bins = n_fft // 2 + 1  # 1025

        # Encoder: STFT magnitude + mask indicator -> feature per frame
        self.encoder = nn.Sequential(
            nn.Linear(self.freq_bins + 1, 512),  # +1 buat mask channel
            nn.SiLU(),
            nn.Linear(512, feature_dim),
            nn.SiLU(),
        )

        # Temporal modeling buat konteks antar frame (penting buat gap)
        self.temporal = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=feature_dim, nhead=8,
                dim_feedforward=512, batch_first=True, dropout=0.1
            ),
            num_layers=4,
        )

        # Decoder: features -> predicted STFT magnitude
        self.mag_decoder = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.SiLU(),
            nn.Linear(512, self.freq_bins),
            nn.ReLU(),  # magnitude selalu non-negative
        )

    def _sample_to_frame_mask(self, mask, n_frames):
        """Konversi sample-level boolean mask -> frame-level float mask."""
        B = mask.shape[0]
        device = mask.device
        frame_mask = torch.zeros(B, n_frames, device=device)
        for i in range(n_frames):
            start = i * self.hop_length
            end = min(start + self.hop_length, mask.shape[-1])
            frame_mask[:, i] = mask[:, start:end].float().mean(dim=-1)
        return frame_mask  # (B, T) values in [0, 1]

    def get_features(self, x: torch.Tensor, mask: torch.Tensor = None):
        """
        Extract temporal features dari audio.

        Args:
            x: audio input (B, N) atau (B, 1, N)
            mask: boolean mask (B, N), True = gap region
        Returns:
            features: (B, T, feature_dim) — sequence of per-frame features
        """
        x_in = x.squeeze(1) if x.dim() == 3 else x

        # STFT magnitude
        spec = torch.stft(
            x_in, n_fft=self.n_fft, hop_length=self.hop_length,
            return_complex=True
        )
        mag = spec.abs().permute(0, 2, 1)  # (B, T, F)
        n_frames = mag.shape[1]

        # Tambah mask channel biar model tahu posisi gap
        if mask is not None:
            frame_mask = self._sample_to_frame_mask(mask, n_frames)
            mag_with_mask = torch.cat([mag, frame_mask.unsqueeze(-1)], dim=-1)
        else:
            zeros = torch.zeros(mag.shape[0], n_frames, 1, device=mag.device)
            mag_with_mask = torch.cat([mag, zeros], dim=-1)

        # Per-frame encoding
        features = self.encoder(mag_with_mask)  # (B, T, feature_dim)

        # Temporal context
        features = self.temporal(features)  # (B, T, feature_dim)
        return features

    def decode_features(self, features: torch.Tensor):
        """Decode conditioned features ke STFT magnitude prediction."""
        return self.mag_decoder(features)  # (B, T, F)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None, conditioning=None):
        """Full forward pass: audio -> predicted STFT magnitude."""
        features = self.get_features(x, mask)
        if conditioning is not None:
            features = features + conditioning.unsqueeze(1)
        return self.decode_features(features)

    def inpaint(self, masked_audio: torch.Tensor, mask: torch.Tensor, conditioning=None):
        """
        Inpainting: prediksi STFT magnitude gap region, lalu iSTFT.

        Args:
            masked_audio: (B, N) atau (1, N) audio dengan gap di-zero-kan
            mask: (B, N) boolean mask, True = gap region
            conditioning: (B, feature_dim) dari FiLM, atau None buat baseline
        Returns:
            numpy array reconstructed audio
        """
        if masked_audio.dim() == 1:
            masked_audio = masked_audio.unsqueeze(0)
        if mask.dim() == 1:
            mask = mask.unsqueeze(0)

        x_in = masked_audio.squeeze(1) if masked_audio.dim() == 3 else masked_audio

        # Prediksi STFT magnitude
        features = self.get_features(x_in, mask)
        if conditioning is not None:
            # conditioning bisa (B, feature_dim) atau (B, T, feature_dim)
            if conditioning.dim() == 2:
                conditioning = conditioning.unsqueeze(1)
            features = features + conditioning
        pred_mag = self.decode_features(features)  # (B, T, F)
        pred_mag = pred_mag.permute(0, 2, 1)  # (B, F, T)

        # Phase dari input audio (buat rekonstruksi)
        input_spec = torch.stft(
            x_in, n_fft=self.n_fft, hop_length=self.hop_length,
            return_complex=True
        )
        input_phase = torch.angle(input_spec)

        # Bikin complex STFT dari predicted magnitude + input phase
        recon_spec = pred_mag * torch.exp(1j * input_phase)

        # iSTFT -> waveform
        reconstructed = torch.istft(
            recon_spec, n_fft=self.n_fft, hop_length=self.hop_length,
            length=x_in.shape[-1]
        )

        # Hanya replace gap region
        output = x_in.clone()
        output[mask.bool()] = reconstructed[mask.bool()]

        if output.shape[0] == 1:
            return output[0].detach().cpu().numpy()
        return output.detach().cpu().numpy()


class AudioMAEProxy(nn.Module):
    def __init__(self, output_dim=768):
        super().__init__()
        self.output_dim = output_dim
        self.patch_embed = nn.Linear(128, 256)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=256, nhead=8, batch_first=True),
            num_layers=4,
        )
        self.proj = nn.Linear(256, output_dim)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.patch_embed(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.proj(x)


class SharedMAIDDecoder(nn.Module):
    """
    MAID replica bersama untuk training dan evaluation hybrid.

    Training menggunakan loss rekonstruksi pada domain mel untuk menjaga scope tetap
    selaras dengan replica yang sudah ada, tanpa memaksa refactor penuh ke DDPM.
    """

    def __init__(self, n_mels: int = 128, feature_dim: int = 512, n_fft: int = 2048, hop_length: int = 512):
        super().__init__()
        self.n_mels = n_mels
        self.feature_dim = feature_dim
        self.n_fft = n_fft
        self.hop_length = hop_length

        self.encoder_blocks = nn.ModuleList([
            nn.Sequential(nn.Linear(n_mels, 512), nn.SiLU()),
            nn.Sequential(nn.Linear(512, 512), nn.SiLU()),
            nn.Sequential(nn.Linear(512, feature_dim), nn.SiLU()),
        ])

        self.decoder_blocks = nn.ModuleList([
            nn.Sequential(nn.Linear(feature_dim, 512), nn.SiLU()),
            nn.Sequential(nn.Linear(512, 512), nn.SiLU()),
            nn.Sequential(nn.Linear(512, n_mels)),
        ])

        self.feature_pool = nn.Sequential(
            nn.Linear(n_mels, feature_dim),
            nn.SiLU(),
        )

    @property
    def device(self):
        return next(self.parameters()).device

    def audio_to_mel_batch(self, audio_batch, sr: int = TARGET_SR):
        mel_db_list = []
        for audio in ensure_audio_list(audio_batch):
            mel = librosa.feature.melspectrogram(
                y=audio,
                sr=sr,
                n_mels=self.n_mels,
                n_fft=self.n_fft,
                hop_length=self.hop_length,
            )
            mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
            mel_db_list.append(mel_db)

        mel_db_tensor = torch.from_numpy(np.stack(mel_db_list, axis=0)).float().to(self.device)
        mel_norm_tensor = (mel_db_tensor + 40.0) / 40.0
        return mel_db_tensor, mel_norm_tensor

    def mask_to_frame_mask(self, mask_batch, frame_count: int):
        if isinstance(mask_batch, torch.Tensor):
            mask_np = mask_batch.detach().cpu().numpy().astype(bool)
        else:
            mask_np = np.asarray(mask_batch).astype(bool)

        if mask_np.ndim == 1:
            mask_np = mask_np[None, :]

        frame_mask = np.zeros((mask_np.shape[0], frame_count), dtype=bool)
        for batch_idx in range(mask_np.shape[0]):
            for frame_idx in range(frame_count):
                start = frame_idx * self.hop_length
                end = min(start + self.hop_length, mask_np.shape[1])
                frame_mask[batch_idx, frame_idx] = np.any(mask_np[batch_idx, start:end])

        return torch.from_numpy(frame_mask).to(self.device)

    def get_features(self, x: torch.Tensor):
        _, mel_norm = self.audio_to_mel_batch(x)
        pooled = mel_norm.mean(dim=-1)
        return self.feature_pool(pooled)

    def predict_mel_norm(self, masked_audio: torch.Tensor, conditioning=None):
        _, mel_norm = self.audio_to_mel_batch(masked_audio)
        x = mel_norm.permute(0, 2, 1)

        for block in self.encoder_blocks:
            x = block(x)

        if conditioning is not None:
            x = x + conditioning.unsqueeze(1)

        for block in self.decoder_blocks:
            x = block(x)

        return x

    def mel_db_to_audio(self, mel_db_pred: np.ndarray, sr: int = TARGET_SR):
        mel_power = np.maximum(librosa.db_to_power(mel_db_pred), 1e-10)
        stft = librosa.feature.inverse.mel_to_stft(
            mel_power,
            sr=sr,
            n_fft=self.n_fft,
            power=2.0,
        )
        return librosa.griffinlim(
            stft,
            n_iter=128,
            hop_length=self.hop_length,
            momentum=0.99,
        )

    def inpaint(self, masked_audio: torch.Tensor, mask: torch.Tensor, conditioning=None, n_steps: int = 50):
        if masked_audio.dim() == 1:
            masked_audio = masked_audio.unsqueeze(0)
        if mask.dim() == 1:
            mask = mask.unsqueeze(0)

        base_mel_db, _ = self.audio_to_mel_batch(masked_audio)
        pred_mel_norm = self.predict_mel_norm(masked_audio, conditioning=conditioning)
        pred_mel_db = pred_mel_norm.permute(0, 2, 1) * 40.0 - 40.0
        gap_frame_mask = self.mask_to_frame_mask(mask, pred_mel_norm.shape[1]).detach().cpu().numpy()

        outputs = []
        base_mel_db_np = base_mel_db.detach().cpu().numpy()
        pred_mel_db_np = pred_mel_db.detach().cpu().numpy()
        mask_np = mask.detach().cpu().numpy().astype(bool)
        masked_audio_np = masked_audio.detach().cpu().numpy()

        for batch_idx in range(masked_audio.shape[0]):
            output_mel_db = base_mel_db_np[batch_idx].copy()
            output_mel_db[:, gap_frame_mask[batch_idx]] = pred_mel_db_np[batch_idx][:, gap_frame_mask[batch_idx]]

            reconstructed = self.mel_db_to_audio(output_mel_db)
            target_len = masked_audio_np.shape[-1]
            if len(reconstructed) > target_len:
                reconstructed = reconstructed[:target_len]
            elif len(reconstructed) < target_len:
                reconstructed = np.pad(reconstructed, (0, target_len - len(reconstructed)))

            output = masked_audio_np[batch_idx].copy()
            output[mask_np[batch_idx]] = reconstructed[mask_np[batch_idx]]
            outputs.append(output)

        if len(outputs) == 1:
            return outputs[0]
        return np.stack(outputs, axis=0)



def ensure_audio_list(audio_batch):
    if isinstance(audio_batch, torch.Tensor):
        audio_batch = audio_batch.detach().cpu().numpy()

    if isinstance(audio_batch, np.ndarray):
        if audio_batch.ndim == 1:
            return [audio_batch.astype(np.float32)]
        return [sample.astype(np.float32) for sample in audio_batch]

    if isinstance(audio_batch, (list, tuple)):
        return [np.asarray(sample, dtype=np.float32) for sample in audio_batch]

    raise TypeError(f"Tipe audio batch tidak didukung: {type(audio_batch)}")



def build_film_layer(model_name: str, device):
    cfg = FILM_CONFIGS[model_name]
    return FiLMLayer(
        encoder_dim=cfg["encoder_dim"],
        decoder_feature_dim=cfg["decoder_feature_dim"],
    ).to(device)



def build_hybrid_cqtdiff_decoder(device):
    decoder = SharedCQTDiffProxyModel().to(device)
    decoder.eval()
    print("ℹ️ Hybrid CQT-Diff+ menggunakan shared proxy decoder untuk konsistensi training/evaluation.")
    return decoder



def build_maid_decoder(device):
    decoder = SharedMAIDDecoder().to(device)
    decoder.eval()
    return decoder



def build_clap_encoder(device):
    from transformers import ClapModel, ClapProcessor

    torch_dtype = torch.float16 if str(device).startswith("cuda") else torch.float32
    processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
    model = ClapModel.from_pretrained(
        "laion/clap-htsat-unfused",
        torch_dtype=torch_dtype,
    ).to(device)
    model.eval()

    def encode(audio_batch, sr: int = TARGET_SR):
        audio_list = []
        for audio in ensure_audio_list(audio_batch):
            if sr != 48000:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=48000)
            audio_list.append(audio.astype(np.float32))

        inputs = processor(audio=audio_list, sampling_rate=48000, return_tensors="pt", padding=True)
        inputs = {
            key: value.to(device=device, dtype=torch_dtype if value.is_floating_point() else value.dtype)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            audio_features = model.get_audio_features(**inputs)

        if isinstance(audio_features, torch.Tensor):
            return audio_features.float()
        if hasattr(audio_features, "pooler_output"):
            return audio_features.pooler_output.float()
        raise TypeError(f"Output CLAP tidak dikenali: {type(audio_features)}")

    return model, encode



def build_audiomae_encoder(device):
    torch_dtype = torch.float16 if str(device).startswith("cuda") else torch.float32

    try:
        from transformers import AutoFeatureExtractor, AutoModel

        model_name = "facebook/audiomae-base-audioset"
        processor = AutoFeatureExtractor.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name, torch_dtype=torch_dtype).to(device)
        model.eval()
        use_real = True
        print("✅ AudioMAE dari HuggingFace berhasil diload.")
    except Exception as exc:
        processor = None
        model = AudioMAEProxy().to(device)
        model.eval()
        use_real = False
        print(f"⚠️ AudioMAE real tidak tersedia, memakai proxy ({exc})")

    def encode(audio_batch, sr: int = TARGET_SR):
        audio_list = ensure_audio_list(audio_batch)

        with torch.inference_mode():
            if use_real:
                inputs = processor(audio_list, sampling_rate=sr, return_tensors="pt", padding=True)
                inputs = {
                    key: value.to(device=device, dtype=torch_dtype if value.is_floating_point() else value.dtype)
                    for key, value in inputs.items()
                }
                outputs = model(**inputs)
                if hasattr(outputs, "last_hidden_state"):
                    return outputs.last_hidden_state[:, 0, :].float()
                if hasattr(outputs, "pooler_output"):
                    return outputs.pooler_output.float()
                raise TypeError(f"Output AudioMAE tidak dikenali: {type(outputs)}")

            mel_batch = []
            for audio in audio_list:
                mel = librosa.feature.melspectrogram(
                    y=audio,
                    sr=sr,
                    n_mels=128,
                    n_fft=2048,
                    hop_length=512,
                )
                mel_db = librosa.power_to_db(mel, ref=np.max)
                mel_norm = (mel_db + 40.0) / 40.0
                mel_batch.append(torch.from_numpy(mel_norm).float())

            mel_tensor = torch.stack(mel_batch, dim=0).to(device)
            return model(mel_tensor).float()

    return model, encode



def train_maid_step(decoder, encoder_fn, film, batch, optimizer, scaler, cfg_drop=0.1, device="cuda"):
    """Training step MAID: reconstruction loss di mel domain."""
    decoder.train()
    film.train()

    clean = batch["clean"].to(device, non_blocking=True)
    masked = batch["masked"].to(device, non_blocking=True)
    mask = batch["mask"].to(device, non_blocking=True)
    batch_size = clean.size(0)

    with torch.no_grad():
        z = encoder_fn(masked)

    # CFG dropout
    drop = (torch.rand(batch_size, device=device) < cfg_drop).view(-1, *([1] * (z.dim() - 1)))
    z = torch.where(drop, torch.zeros_like(z), z)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
        # MAID get_features output (B, feature_dim) — 2D
        decoder_features = decoder.get_features(masked)
        conditioned_features = film(z.float(), decoder_features)
        pred_mel_norm = decoder.predict_mel_norm(masked, conditioning=conditioned_features)
        _, clean_mel_norm = decoder.audio_to_mel_batch(clean)
        clean_mel_norm = clean_mel_norm.permute(0, 2, 1)

        frame_mask = decoder.mask_to_frame_mask(mask, pred_mel_norm.shape[1])
        expanded_mask = frame_mask.unsqueeze(-1).expand_as(pred_mel_norm)

        gap_loss = F.l1_loss(pred_mel_norm[expanded_mask], clean_mel_norm[expanded_mask])
        full_loss = F.l1_loss(pred_mel_norm, clean_mel_norm)
        loss = gap_loss + 0.1 * full_loss

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        list(decoder.parameters()) + list(film.parameters()), max_norm=1.0
    )
    scaler.step(optimizer)
    scaler.update()

    return {
        "loss": loss.item(),
        "gap_loss": gap_loss.item(),
        "full_loss": full_loss.item(),
    }


def train_maid_model(decoder, encoder_fn, film, train_loader, val_loader=None,
                     num_epochs=20, lr=1e-4, device="cuda", checkpoint_dir=None,
                     model_name="model"):
    """Training loop MAID (mel reconstruction)."""
    optimizer = torch.optim.AdamW(
        list(decoder.parameters()) + list(film.parameters()),
        lr=lr, weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    best_val_loss = float("inf")
    checkpoint_path = None
    if checkpoint_dir:
        os.makedirs(checkpoint_dir, exist_ok=True)
        checkpoint_path = os.path.join(checkpoint_dir, f"{model_name}_best.pt")

    for epoch in range(num_epochs):
        decoder.train()
        film.train()
        epoch_losses = []

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            metrics = train_maid_step(
                decoder, encoder_fn, film, batch, optimizer, scaler, device=device,
            )
            epoch_losses.append(metrics["loss"])

        avg_loss = float(np.mean(epoch_losses))
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.6f} | LR: {lr_now:.2e}")
        scheduler.step()

        if val_loader is not None and (epoch + 1) % 5 == 0:
            decoder.eval()
            film.eval()
            val_losses = []

            with torch.inference_mode():
                for batch in val_loader:
                    clean = batch["clean"].to(device)
                    masked = batch["masked"].to(device)
                    mask_b = batch["mask"].to(device)

                    z = encoder_fn(masked)
                    decoder_features = decoder.get_features(masked)
                    conditioned_features = film(z.float(), decoder_features)
                    pred_mel_norm = decoder.predict_mel_norm(masked, conditioning=conditioned_features)
                    _, clean_mel_norm = decoder.audio_to_mel_batch(clean)
                    clean_mel_norm = clean_mel_norm.permute(0, 2, 1)
                    frame_mask = decoder.mask_to_frame_mask(mask_b, pred_mel_norm.shape[1])
                    expanded_mask = frame_mask.unsqueeze(-1).expand_as(pred_mel_norm)
                    val_loss = F.l1_loss(pred_mel_norm[expanded_mask], clean_mel_norm[expanded_mask])
                    val_losses.append(val_loss.item())

            avg_val = float(np.mean(val_losses))
            print(f"  Val Loss: {avg_val:.6f}")

            if checkpoint_path and avg_val < best_val_loss:
                best_val_loss = avg_val
                torch.save({
                    "epoch": epoch,
                    "decoder_state": decoder.state_dict(),
                    "film_state": film.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "val_loss": avg_val,
                }, checkpoint_path)
                print(f"  💾 Best checkpoint saved: {checkpoint_path}")

    if checkpoint_path and not os.path.exists(checkpoint_path):
        torch.save({
            "epoch": num_epochs - 1,
            "decoder_state": decoder.state_dict(),
            "film_state": film.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": best_val_loss,
        }, checkpoint_path)
        print(f"  💾 Fallback checkpoint saved: {checkpoint_path}")

    print(f"\n✅ Training MAID selesai! Best val loss: {best_val_loss:.6f}")
    return decoder, film


def run_hybrid_inpainting_evaluation(model_label: str, encoder_fn, decoder, film_layer, device, n_eval_samples: int = 50):
    """
    Evaluasi hybrid model (encoder + FiLM + decoder).

    PERBAIKAN: sekarang pass mask ke get_features() untuk CQT-Diff+ proxy,
    dan FiLM conditioning di-inject dengan benar ke sequence features.
    """
    original_audios, masked_by_gap = load_preprocessed_data(n_eval_samples)
    reconstructed_dict = {}

    print(f"\n🎵 Menjalankan inpainting {model_label}...")

    for gap_ms in GAP_DURATIONS_MS:
        print(f"\n  Gap {gap_ms}ms...")
        reconstructed_list = []

        for index, (orig_audio, masked_audio) in enumerate(zip(original_audios, masked_by_gap[gap_ms])):
            if (index + 1) % 10 == 0:
                print(f"    Sample {index+1}/{n_eval_samples}")

            with torch.inference_mode():
                # Encode masked audio pakai SSL encoder
                encoder_latent = encoder_fn(masked_audio)

                masked_tensor = torch.from_numpy(masked_audio).float().unsqueeze(0).to(device)
                mask, gap_start, gap_end = make_gap_mask(len(masked_audio), gap_ms)
                mask_tensor = torch.from_numpy(mask).unsqueeze(0).to(device)

                # Cek apakah decoder punya mask-aware get_features (CQT-Diff+ proxy baru)
                import inspect
                sig = inspect.signature(decoder.get_features)
                if 'mask' in sig.parameters:
                    # CQT-Diff+ proxy baru: get_features(x, mask) -> (B, T, D)
                    decoder_features = decoder.get_features(masked_tensor, mask_tensor)
                    conditioned_features = film_layer(encoder_latent.float(), decoder_features)
                else:
                    # MAID: get_features(x) -> (B, D)
                    decoder_features = decoder.get_features(masked_tensor)
                    conditioned_features = film_layer(encoder_latent.float(), decoder_features)

                # Inpaint dengan conditioned features
                reconstructed = decoder.inpaint(
                    masked_tensor,
                    mask_tensor,
                    conditioning=conditioned_features,
                )

                # Crossfade buat menghilangkan click artifacts
                reconstructed = crossfade_boundary(
                    orig_audio, reconstructed, gap_start, gap_end,
                )

            reconstructed_list.append(reconstructed)

        reconstructed_dict[gap_ms] = reconstructed_list

    return evaluate_all_gaps(original_audios, reconstructed_dict, TARGET_SR)


def run_baseline_inpainting_evaluation(decoder, device, n_eval_samples: int = 50):
    """
    Evaluasi baseline (tanpa encoder, tanpa FiLM).
    Decoder dipakai langsung tanpa conditioning.
    """
    original_audios, masked_by_gap = load_preprocessed_data(n_eval_samples)
    reconstructed_dict = {}

    print("\n🎵 Menjalankan baseline CQT-Diff+ (tanpa SSL encoder)...")

    for gap_ms in GAP_DURATIONS_MS:
        print(f"\n  Gap {gap_ms}ms...")
        reconstructed_list = []

        for i, (orig_audio, masked_audio) in enumerate(zip(original_audios, masked_by_gap[gap_ms])):
            if (i + 1) % 10 == 0:
                print(f"    Sample {i+1}/{n_eval_samples}")

            with torch.inference_mode():
                masked_tensor = torch.from_numpy(masked_audio).float().unsqueeze(0).to(device)
                mask, gap_start, gap_end = make_gap_mask(len(masked_audio), gap_ms)
                mask_tensor = torch.from_numpy(mask).unsqueeze(0).to(device)

                # Baseline: conditioning=None
                reconstructed = decoder.inpaint(
                    masked_tensor, mask_tensor, conditioning=None,
                )
                reconstructed = crossfade_boundary(
                    orig_audio, reconstructed, gap_start, gap_end,
                )

            reconstructed_list.append(reconstructed)

        reconstructed_dict[gap_ms] = reconstructed_list

    return evaluate_all_gaps(original_audios, reconstructed_dict, TARGET_SR)


print("✅ Shared hybrid training helpers berhasil didefinisikan!")
print("   Tersedia: SharedCQTDiffProxyModel (temporal), SharedMAIDDecoder,")
print("   builder encoder/decoder, checkpoint helpers, trainer MAID,")
print("   run_hybrid_inpainting_evaluation, run_baseline_inpainting_evaluation")

---
## CELL 7 — BASELINE: CQT-Diff+ Standalone (Trained)

Ini adalah **baseline** — CQT-Diff+ proxy **di-train tanpa encoder SSL dan tanpa FiLM**.

**Perubahan penting dari versi sebelumnya:**
- Baseline sekarang **juga di-train** (reconstruction loss di STFT domain)
- Perbandingan menjadi **fair**: satu-satunya perbedaan antara baseline vs hybrid adalah **ada/tidaknya SSL encoder conditioning**
- Bukan lagi random weights vs trained

Fungsinya sebagai titik pembanding:
- Kalau model hybrid (Cell 8-11) lebih baik dari baseline → SSL encoder memberikan kontribusi nyata
- Kalau hasil hampir sama → perlu investigasi lebih lanjut

⚠️ **Jalankan cell ini sebelum Cell 8-11.**

In [ ]:
# ============================================================
# CELL 7: BASELINE — CQT-Diff+ STANDALONE (TRAINED)
# ============================================================
# Baseline = CQT-Diff+ proxy TRAINED tanpa encoder SSL dan tanpa FiLM.
#
# PERBAIKAN dari versi sebelumnya:
# - Baseline sekarang JUGA DI-TRAIN (tanpa encoder conditioning)
# - Perbandingan jadi FAIR: bedanya hybrid vs baseline cuma ada/tidaknya
#   SSL encoder conditioning, bukan trained vs untrained
# - Pakai arsitektur SharedCQTDiffProxyModel yang sama dengan hybrid
# - Training pakai reconstruction loss (bukan random weights)
#
# Alur:
# 1. Cek apakah baseline checkpoint sudah ada
# 2. Kalau belum: train baseline (tanpa encoder) -> simpan checkpoint
# 3. Load checkpoint -> evaluasi
# ============================================================

import torch
import numpy as np
import os

MODEL_NAME = "baseline_cqtdiff"
FORCE_RETRAIN = True    # <-- True karena arsitektur model berubah total!
FORCE_REEVAL = True     # <-- True buat hapus hasil lama dan re-evaluasi
NUM_EPOCHS = 10
BATCH_SIZE = 4
NUM_WORKERS = 0
LEARNING_RATE = 1e-4

# Hapus hasil lama kalau FORCE_REEVAL aktif (biar check_if_done gak skip)
if FORCE_REEVAL:
    old_result = os.path.join(PATHS["results"], f"{MODEL_NAME}_results.csv")
    if os.path.exists(old_result):
        os.remove(old_result)
        print(f"🗑️ Hasil lama dihapus: {old_result}")

# Hapus checkpoint lama yang inkompatibel dengan arsitektur baru
if FORCE_RETRAIN:
    old_ckpt = os.path.join(get_model_checkpoint_dir(MODEL_NAME), f"{MODEL_NAME}_best.pt")
    if os.path.exists(old_ckpt):
        os.remove(old_ckpt)
        print(f"🗑️ Checkpoint lama (inkompatibel) dihapus: {old_ckpt}")

if check_if_done(MODEL_NAME):
    print(f"Baseline {MODEL_NAME} sudah selesai. Lewati cell ini.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🔧 Device: {device}")
    print_gpu_usage("Awal")

    # ============================================================
    # TRAINING BASELINE (tanpa encoder, tanpa FiLM)
    # ============================================================
    ckpt_dir = get_model_checkpoint_dir(MODEL_NAME)
    ckpt_path = os.path.join(ckpt_dir, f"{MODEL_NAME}_best.pt")

    if os.path.exists(ckpt_path) and not FORCE_RETRAIN:
        print(f"✅ Baseline checkpoint sudah ada: {ckpt_path}")
    else:
        print("\n🏋️ Training baseline CQT-Diff+ (tanpa encoder)...")
        print("   Ini biar perbandingan fair: baseline juga trained,")
        print("   bedanya cuma TANPA SSL encoder conditioning.\n")

        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
        baseline_model = SharedCQTDiffProxyModel().to(device)
        print_gpu_usage("Sebelum training baseline")

        train_baseline_model(
            baseline_model,
            loaders["train"],
            val_loader=loaders["val"],
            num_epochs=NUM_EPOCHS,
            lr=LEARNING_RATE,
            device=device,
            checkpoint_dir=ckpt_dir,
            model_name=MODEL_NAME,
        )
        del baseline_model
        clear_gpu_memory()

    # ============================================================
    # EVALUASI BASELINE
    # ============================================================
    print("\n📥 Loading trained baseline untuk evaluasi...")
    baseline_model = SharedCQTDiffProxyModel().to(device)
    load_baseline_checkpoint(baseline_model, device)
    baseline_model.eval()
    print_gpu_usage("Setelah load baseline")

    # Evaluasi pakai fungsi shared
    print("\n📊 Mengevaluasi baseline...")
    results_df = run_baseline_inpainting_evaluation(baseline_model, device, n_eval_samples=50)

    print(f"\n📋 Hasil evaluasi BASELINE (CQT-Diff+ standalone, trained tanpa encoder):")
    print(results_df.to_string(index=False))

    save_results(results_df, MODEL_NAME)

    # UNLOAD
    print("\n🧹 Membersihkan memori GPU...")
    clear_gpu_memory(baseline_model)

    print(f"\n✅ BASELINE {MODEL_NAME} selesai!")

---
## CELL 8 — Kombinasi 1: CLAP + CQT-Diff+

Model hybrid pertama. Dibandingkan dengan baseline di Cell 7,
perbedaannya hanya penambahan **CLAP encoder + FiLM conditioning**.

In [ ]:
# ============================================================
# CELL 8A: TRAINING — CLAP + CQT-Diff+
# ============================================================
# Default: skip training jika checkpoint sudah ada.
# Set FORCE_RETRAIN = True untuk melatih ulang.
# ============================================================

# ============================================================
# CELL 8A: TRAINING — CLAP + CQT-Diff+
# ============================================================
# Set FORCE_RETRAIN = True untuk melatih ulang.
# PENTING: Set True setelah update arsitektur model!
# ============================================================

MODEL_NAME = "clap_cqtdiff"
FORCE_RETRAIN = True   # <-- True karena arsitektur model berubah!
BATCH_SIZE = 4
NUM_WORKERS = 0
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

assert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."

ckpt_path = get_model_checkpoint_path(MODEL_NAME)
if hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:
    print(f"✅ Checkpoint sudah ada. Skip training: {ckpt_path}")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clap_model = None
    cqtdiff_model = None
    film_layer = None

    try:
        print(f"🔧 Device: {device}")
        if FORCE_RETRAIN and hybrid_checkpoint_exists(MODEL_NAME):
            print(f"♻️ FORCE_RETRAIN aktif. Checkpoint lama akan ditimpa: {ckpt_path}")

        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
        clap_model, encoder_fn = build_clap_encoder(device)
        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)
        film_layer = build_film_layer(MODEL_NAME, device)

        print_gpu_usage("Sebelum training")
        train_model(
            cqtdiff_model,
            encoder_fn,
            film_layer,
            loaders["train"],
            val_loader=loaders["val"],
            num_epochs=NUM_EPOCHS,
            lr=LEARNING_RATE,
            device=device,
            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),
            model_name=MODEL_NAME,
        )

        if not hybrid_checkpoint_exists(MODEL_NAME):
            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")

        print(f"✅ Training selesai. Checkpoint siap dipakai: {ckpt_path}")
    finally:
        clear_gpu_memory(clap_model, cqtdiff_model, film_layer)

In [ ]:
# ============================================================
# CELL 8: KOMBINASI 1 — CLAP + CQT-Diff+
# ============================================================
# Evaluasi hybrid selalu memakai checkpoint terlatih.
# Jika checkpoint belum ada, jalankan CELL 8A terlebih dahulu.
# ============================================================

MODEL_NAME = "clap_cqtdiff"
FORCE_REEVAL = True  # <-- True buat re-evaluasi setelah update arsitektur

if FORCE_REEVAL:
    old_result = os.path.join(PATHS["results"], f"{MODEL_NAME}_results.csv")
    if os.path.exists(old_result):
        os.remove(old_result)

if check_if_done(MODEL_NAME):
    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")
else:
    ckpt_path = get_model_checkpoint_path(MODEL_NAME)
    if not hybrid_checkpoint_exists(MODEL_NAME):
        raise FileNotFoundError(
            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. Jalankan CELL 8A terlebih dahulu."
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clap_model = None
    cqtdiff_model = None
    film_layer = None

    try:
        print_gpu_usage("Awal")
        print(f"📦 Menggunakan checkpoint: {ckpt_path}")

        clap_model, encoder_fn = build_clap_encoder(device)
        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)
        film_layer = build_film_layer(MODEL_NAME, device)
        load_hybrid_checkpoint(MODEL_NAME, cqtdiff_model, film_layer, device)
        print_gpu_usage("Setelah load model terlatih")

        results_df = run_hybrid_inpainting_evaluation(
            "CLAP + CQT-Diff+",
            encoder_fn,
            cqtdiff_model,
            film_layer,
            device,
            n_eval_samples=50,
        )

        print(f"\n📋 Hasil {MODEL_NAME}:")
        print(results_df.to_string(index=False))
        save_results(results_df, MODEL_NAME)
    finally:
        clear_gpu_memory(clap_model, cqtdiff_model, film_layer)

    print(f"\n✅ {MODEL_NAME} selesai!")

---
## CELL 9 — Kombinasi 2: CLAP + MAID

In [ ]:
# ============================================================
# CELL 9A: TRAINING — CLAP + MAID
# ============================================================
# Default: skip training jika checkpoint sudah ada.
# Set FORCE_RETRAIN = True untuk melatih ulang.
# ============================================================

MODEL_NAME = "clap_maid"
FORCE_RETRAIN = True   # <-- True karena arsitektur/training berubah!
BATCH_SIZE = 4
NUM_WORKERS = 0
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

assert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."

ckpt_path = get_model_checkpoint_path(MODEL_NAME)
if hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:
    print(f"✅ Checkpoint sudah ada. Skip training: {ckpt_path}")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clap_model = None
    maid_model = None
    film_layer = None

    try:
        print(f"🔧 Device: {device}")
        if FORCE_RETRAIN and hybrid_checkpoint_exists(MODEL_NAME):
            print(f"♻️ FORCE_RETRAIN aktif. Checkpoint lama akan ditimpa: {ckpt_path}")

        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
        clap_model, encoder_fn = build_clap_encoder(device)
        maid_model = build_maid_decoder(device)
        film_layer = build_film_layer(MODEL_NAME, device)

        print_gpu_usage("Sebelum training")
        train_maid_model(
            maid_model,
            encoder_fn,
            film_layer,
            loaders["train"],
            val_loader=loaders["val"],
            num_epochs=NUM_EPOCHS,
            lr=LEARNING_RATE,
            device=device,
            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),
            model_name=MODEL_NAME,
        )

        if not hybrid_checkpoint_exists(MODEL_NAME):
            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")

        print(f"✅ Training selesai. Checkpoint siap dipakai: {ckpt_path}")
    finally:
        clear_gpu_memory(clap_model, maid_model, film_layer)

In [ ]:
# ============================================================
# CELL 9: KOMBINASI 2 — CLAP + MAID
# ============================================================
# Evaluasi hybrid selalu memakai checkpoint terlatih.
# Jika checkpoint belum ada, jalankan CELL 9A terlebih dahulu.
# ============================================================

MODEL_NAME = "clap_maid"
FORCE_REEVAL = True  # <-- True buat re-evaluasi setelah update

if FORCE_REEVAL:
    old_result = os.path.join(PATHS["results"], f"{MODEL_NAME}_results.csv")
    if os.path.exists(old_result):
        os.remove(old_result)

if check_if_done(MODEL_NAME):
    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")
else:
    ckpt_path = get_model_checkpoint_path(MODEL_NAME)
    if not hybrid_checkpoint_exists(MODEL_NAME):
        raise FileNotFoundError(
            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. Jalankan CELL 9A terlebih dahulu."
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clap_model = None
    maid_model = None
    film_layer = None

    try:
        print_gpu_usage("Awal")
        print(f"📦 Menggunakan checkpoint: {ckpt_path}")

        clap_model, encoder_fn = build_clap_encoder(device)
        maid_model = build_maid_decoder(device)
        film_layer = build_film_layer(MODEL_NAME, device)
        load_hybrid_checkpoint(MODEL_NAME, maid_model, film_layer, device)
        print_gpu_usage("Setelah load model terlatih")

        results_df = run_hybrid_inpainting_evaluation(
            "CLAP + MAID",
            encoder_fn,
            maid_model,
            film_layer,
            device,
            n_eval_samples=50,
        )

        print(f"\n📋 Hasil {MODEL_NAME}:")
        print(results_df.to_string(index=False))
        save_results(results_df, MODEL_NAME)
    finally:
        clear_gpu_memory(clap_model, maid_model, film_layer)

    print(f"\n✅ {MODEL_NAME} selesai!")

---
## CELL 10 — Kombinasi 3: AudioMAE + CQT-Diff+

In [ ]:
# ============================================================
# CELL 10A: TRAINING — AudioMAE + CQT-Diff+
# ============================================================
# Default: skip training jika checkpoint sudah ada.
# Set FORCE_RETRAIN = True untuk melatih ulang.
# ============================================================

MODEL_NAME = "audiomae_cqtdiff"
FORCE_RETRAIN = True   # <-- True karena arsitektur model berubah!
BATCH_SIZE = 4
NUM_WORKERS = 0
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

assert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."

ckpt_path = get_model_checkpoint_path(MODEL_NAME)
if hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:
    print(f"✅ Checkpoint sudah ada. Skip training: {ckpt_path}")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    audiomae_model = None
    cqtdiff_model = None
    film_layer = None

    try:
        print(f"🔧 Device: {device}")
        if FORCE_RETRAIN and hybrid_checkpoint_exists(MODEL_NAME):
            print(f"♻️ FORCE_RETRAIN aktif. Checkpoint lama akan ditimpa: {ckpt_path}")

        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
        audiomae_model, encoder_fn = build_audiomae_encoder(device)
        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)
        film_layer = build_film_layer(MODEL_NAME, device)

        print_gpu_usage("Sebelum training")
        train_model(
            cqtdiff_model,
            encoder_fn,
            film_layer,
            loaders["train"],
            val_loader=loaders["val"],
            num_epochs=NUM_EPOCHS,
            lr=LEARNING_RATE,
            device=device,
            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),
            model_name=MODEL_NAME,
        )

        if not hybrid_checkpoint_exists(MODEL_NAME):
            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")

        print(f"✅ Training selesai. Checkpoint siap dipakai: {ckpt_path}")
    finally:
        clear_gpu_memory(audiomae_model, cqtdiff_model, film_layer)

In [ ]:
# ============================================================
# CELL 10: KOMBINASI 3 — AudioMAE + CQT-Diff+
# ============================================================
# Evaluasi hybrid selalu memakai checkpoint terlatih.
# Jika checkpoint belum ada, jalankan CELL 10A terlebih dahulu.
# ============================================================

MODEL_NAME = "audiomae_cqtdiff"
FORCE_REEVAL = True  # <-- True buat re-evaluasi setelah update arsitektur

if FORCE_REEVAL:
    old_result = os.path.join(PATHS["results"], f"{MODEL_NAME}_results.csv")
    if os.path.exists(old_result):
        os.remove(old_result)

if check_if_done(MODEL_NAME):
    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")
else:
    ckpt_path = get_model_checkpoint_path(MODEL_NAME)
    if not hybrid_checkpoint_exists(MODEL_NAME):
        raise FileNotFoundError(
            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. Jalankan CELL 10A terlebih dahulu."
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    audiomae_model = None
    cqtdiff_model = None
    film_layer = None

    try:
        print_gpu_usage("Awal")
        print(f"📦 Menggunakan checkpoint: {ckpt_path}")

        audiomae_model, encoder_fn = build_audiomae_encoder(device)
        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)
        film_layer = build_film_layer(MODEL_NAME, device)
        load_hybrid_checkpoint(MODEL_NAME, cqtdiff_model, film_layer, device)
        print_gpu_usage("Setelah load model terlatih")

        results_df = run_hybrid_inpainting_evaluation(
            "AudioMAE + CQT-Diff+",
            encoder_fn,
            cqtdiff_model,
            film_layer,
            device,
            n_eval_samples=50,
        )

        print(f"\n📋 Hasil {MODEL_NAME}:")
        print(results_df.to_string(index=False))
        save_results(results_df, MODEL_NAME)
    finally:
        clear_gpu_memory(audiomae_model, cqtdiff_model, film_layer)

    print(f"\n✅ {MODEL_NAME} selesai!")

---
## CELL 11 — Kombinasi 4: AudioMAE + MAID

In [ ]:
# ============================================================
# CELL 11A: TRAINING — AudioMAE + MAID
# ============================================================
# Default: skip training jika checkpoint sudah ada.
# Set FORCE_RETRAIN = True untuk melatih ulang.
# ============================================================

MODEL_NAME = "audiomae_maid"
FORCE_RETRAIN = True   # <-- True karena arsitektur/training berubah!
BATCH_SIZE = 4
NUM_WORKERS = 0
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

assert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."

ckpt_path = get_model_checkpoint_path(MODEL_NAME)
if hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:
    print(f"✅ Checkpoint sudah ada. Skip training: {ckpt_path}")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    audiomae_model = None
    maid_model = None
    film_layer = None

    try:
        print(f"🔧 Device: {device}")
        if FORCE_RETRAIN and hybrid_checkpoint_exists(MODEL_NAME):
            print(f"♻️ FORCE_RETRAIN aktif. Checkpoint lama akan ditimpa: {ckpt_path}")

        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
        audiomae_model, encoder_fn = build_audiomae_encoder(device)
        maid_model = build_maid_decoder(device)
        film_layer = build_film_layer(MODEL_NAME, device)

        print_gpu_usage("Sebelum training")
        train_maid_model(
            maid_model,
            encoder_fn,
            film_layer,
            loaders["train"],
            val_loader=loaders["val"],
            num_epochs=NUM_EPOCHS,
            lr=LEARNING_RATE,
            device=device,
            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),
            model_name=MODEL_NAME,
        )

        if not hybrid_checkpoint_exists(MODEL_NAME):
            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")

        print(f"✅ Training selesai. Checkpoint siap dipakai: {ckpt_path}")
    finally:
        clear_gpu_memory(audiomae_model, maid_model, film_layer)

In [ ]:
# ============================================================
# CELL 11: KOMBINASI 4 — AudioMAE + MAID
# ============================================================
# Evaluasi hybrid selalu memakai checkpoint terlatih.
# Jika checkpoint belum ada, jalankan CELL 11A terlebih dahulu.
# ============================================================

MODEL_NAME = "audiomae_maid"
FORCE_REEVAL = True  # <-- True buat re-evaluasi setelah update

if FORCE_REEVAL:
    old_result = os.path.join(PATHS["results"], f"{MODEL_NAME}_results.csv")
    if os.path.exists(old_result):
        os.remove(old_result)

if check_if_done(MODEL_NAME):
    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")
else:
    ckpt_path = get_model_checkpoint_path(MODEL_NAME)
    if not hybrid_checkpoint_exists(MODEL_NAME):
        raise FileNotFoundError(
            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. Jalankan CELL 11A terlebih dahulu."
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    audiomae_model = None
    maid_model = None
    film_layer = None

    try:
        print_gpu_usage("Awal")
        print(f"📦 Menggunakan checkpoint: {ckpt_path}")

        audiomae_model, encoder_fn = build_audiomae_encoder(device)
        maid_model = build_maid_decoder(device)
        film_layer = build_film_layer(MODEL_NAME, device)
        load_hybrid_checkpoint(MODEL_NAME, maid_model, film_layer, device)
        print_gpu_usage("Setelah load model terlatih")

        results_df = run_hybrid_inpainting_evaluation(
            "AudioMAE + MAID",
            encoder_fn,
            maid_model,
            film_layer,
            device,
            n_eval_samples=50,
        )

        print(f"\n📋 Hasil {MODEL_NAME}:")
        print(results_df.to_string(index=False))
        save_results(results_df, MODEL_NAME)
    finally:
        clear_gpu_memory(audiomae_model, maid_model, film_layer)

    print(f"\n✅ {MODEL_NAME} selesai!")

---
## CELL 12 — Gabungkan & Visualisasikan Semua Hasil

Jalankan setelah baseline dan seluruh kombinasi hybrid selesai dievaluasi.
Untuk hybrid, pastikan cell training dan cell evaluasinya sudah dijalankan sehingga checkpoint dan CSV hasil tersedia.
Grafik akan menampilkan **baseline vs 4 model hybrid** untuk perbandingan langsung.

In [ ]:
# ============================================================
# CELL 12: VISUALISASI HASIL LENGKAP
# ============================================================
# Menampilkan perbandingan semua model:
# - Baseline: CQT-Diff+ standalone (garis putus-putus)
# - 4 kombinasi hybrid (garis solid)
#
# Gap durations: 100, 300, 500, 750, 1200, 1700 ms
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import os

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 10

master_path = os.path.join(PATHS["results"], "all_results.csv")

if not os.path.exists(master_path):
    print("❌ File hasil belum ada. Pastikan Cell 7-11 sudah dijalankan.")
else:
    all_results = pd.read_csv(master_path)

    # Backward compatibility untuk file hasil lama.
    if "PEAQ_ODG" not in all_results.columns:
        if "PEAQ" in all_results.columns:
            all_results["PEAQ_ODG"] = all_results["PEAQ"]
        elif "ODG" in all_results.columns:
            all_results["PEAQ_ODG"] = all_results["ODG"]

    # Cek model mana yang sudah selesai
    available_models = all_results["model"].unique()
    print(f"📊 Model yang tersedia: {list(available_models)}")

    gap_durations = sorted(all_results["gap_ms"].unique())

    # ============================================================
    # TABEL PERBANDINGAN
    # ============================================================
    print("\n" + "="*70)
    print("TABEL PERBANDINGAN LENGKAP")
    print("="*70)

    metric_order = [m for m in ["LSD", "FAD", "PEAQ_ODG"] if m in all_results.columns]
    for metric in metric_order:
        if metric in ["LSD", "FAD"]:
            direction = "↓ lebih rendah = lebih baik"
        else:
            direction = "↑ mendekati 0 = lebih baik"
        print(f"\n{metric} ({direction}):")
        pivot = all_results.pivot(index="gap_ms", columns="model", values=metric)
        # Urutkan kolom: baseline dulu, lalu hybrid
        ordered_cols = [c for c in ["baseline_cqtdiff", "clap_cqtdiff", "clap_maid",
                                     "audiomae_cqtdiff", "audiomae_maid"] if c in pivot.columns]
        print(pivot[ordered_cols].to_string())


    # ============================================================
    # VISUALISASI
    # ============================================================
    # Style per model
    # Baseline: garis putus-putus hitam untuk mudah dibedakan
    # Hybrid: garis solid berwarna
    styles = {
        "baseline_cqtdiff":  {"color": "#000000", "marker": "x", "linestyle": "--",
                               "label": "Baseline: CQT-Diff+", "linewidth": 2.5, "zorder": 10},
        "clap_cqtdiff":      {"color": "#2196F3", "marker": "o", "linestyle": "-",
                               "label": "CLAP + CQT-Diff+", "linewidth": 1.5, "zorder": 5},
        "clap_maid":         {"color": "#4CAF50", "marker": "s", "linestyle": "-",
                               "label": "CLAP + MAID", "linewidth": 1.5, "zorder": 5},
        "audiomae_cqtdiff":  {"color": "#FF9800", "marker": "^", "linestyle": "-",
                               "label": "AudioMAE + CQT-Diff+", "linewidth": 1.5, "zorder": 5},
        "audiomae_maid":     {"color": "#F44336", "marker": "D", "linestyle": "-",
                               "label": "AudioMAE + MAID", "linewidth": 1.5, "zorder": 5},
    }

    metrics_info = {
        "LSD": {"title": "Log Spectral Distance (LSD)",
                "ylabel": "LSD (dB)",
                "note": "↓ lebih rendah = lebih baik"},
        "FAD": {"title": "Frechet Audio Distance (FAD)",
                "ylabel": "FAD Score",
                "note": "↓ lebih rendah = lebih baik"},
        "PEAQ_ODG": {"title": "PEAQ Objective Difference Grade",
                     "ylabel": "PEAQ_ODG Score",
                     "note": "↑ mendekati 0 = lebih baik"},
    }
    metrics_info = {k: v for k, v in metrics_info.items() if k in metric_order}

    fig, axes = plt.subplots(1, len(metrics_info), figsize=(6 * len(metrics_info), 6))
    if len(metrics_info) == 1:
        axes = [axes]
    fig.suptitle(
        "Music Audio Inpainting — Baseline vs Hybrid SSL+Diffusion Models\n"
        f"Gap Durations: {gap_durations} ms",
        fontsize=13, fontweight='bold'
    )

    for ax, (metric, info) in zip(axes, metrics_info.items()):
        # Plot baseline dan hybrid
        # Urutan plot: hybrid dulu, baseline paling atas (zorder lebih tinggi)
        plot_order = [m for m in ["clap_cqtdiff", "clap_maid", "audiomae_cqtdiff",
                                   "audiomae_maid", "baseline_cqtdiff"] if m in available_models]

        for model_name in plot_order:
            model_data = all_results[all_results["model"] == model_name].sort_values("gap_ms")
            s = styles.get(model_name, {"color": "gray", "marker": "x",
                                         "linestyle": "-", "label": model_name,
                                         "linewidth": 1.5, "zorder": 1})
            ax.plot(
                model_data["gap_ms"],
                model_data[metric],
                color=s["color"],
                marker=s["marker"],
                linestyle=s["linestyle"],
                label=s["label"],
                linewidth=s["linewidth"],
                markersize=7,
                zorder=s["zorder"]
            )

        ax.set_title(info["title"], fontsize=11, fontweight='bold')
        ax.set_xlabel("Gap Duration (ms)", fontsize=10)
        ax.set_ylabel(info["ylabel"], fontsize=10)
        ax.set_xticks(gap_durations)
        ax.set_xticklabels([str(g) for g in gap_durations], rotation=45)
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)
        ax.text(0.02, 0.98, info["note"],
                transform=ax.transAxes, fontsize=8,
                verticalalignment='top', style='italic', color='gray')

    plt.tight_layout()

    # Simpan grafik
    plot_path = os.path.join(PATHS["results"], "comparison_plot.png")
    plt.savefig(plot_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"\n💾 Grafik disimpan: {plot_path}")


    # ============================================================
    # RANGKUMAN: IMPROVEMENT HYBRID vs BASELINE
    # ============================================================
    if "baseline_cqtdiff" in available_models:
        print("\n" + "="*72)
        print("📈 IMPROVEMENT HYBRID vs BASELINE (per gap duration)")
        print("   Positif = lebih baik dari baseline")
        print("="*72)

        baseline_data = all_results[all_results["model"] == "baseline_cqtdiff"]

        for gap_ms in gap_durations:
            bl = baseline_data[baseline_data["gap_ms"] == gap_ms].iloc[0]
            print(f"\n  Gap {gap_ms}ms:")

            header_cols = ["Model", "ΔLSD"]
            if "FAD" in all_results.columns:
                header_cols.append("ΔFAD")
            if "PEAQ_ODG" in all_results.columns:
                header_cols.append("ΔPEAQ_ODG")
            print(f"  {header_cols[0]:<25} " + " ".join(f"{h:>10}" for h in header_cols[1:]))
            print(f"  {'-'*65}")

            for model_name in ["clap_cqtdiff", "clap_maid", "audiomae_cqtdiff", "audiomae_maid"]:
                if model_name not in available_models:
                    continue
                hybrid = all_results[
                    (all_results["model"] == model_name) &
                    (all_results["gap_ms"] == gap_ms)
                ].iloc[0]

                # Positif selalu berarti hybrid lebih baik.
                deltas = [bl["LSD"] - hybrid["LSD"]]
                if "FAD" in all_results.columns:
                    deltas.append(bl["FAD"] - hybrid["FAD"])
                if "PEAQ_ODG" in all_results.columns:
                    deltas.append(hybrid["PEAQ_ODG"] - bl["PEAQ_ODG"])

                print(f"  {model_name:<25} " + " ".join(f"{d:>+10.4f}" for d in deltas))

    print(f"\n✅ Semua hasil tersimpan di: {PATHS['results']}")